# Distilling Qwen2.5-0.5B into a 51M-parameter student (GQA + QK-norm, Muon, 2xT4)

**What this notebook does**

| `RUN_MODE` | needs | what happens |
|---|---|---|
| `"data"` | CPU only (no GPU quota) | downloads + tokenizes the **~1B-token training mix** (65% Cosmopedia v2, 35% quality-filtered FineWeb-Edu), three held-out sets, the instruction data, the benchmark sets; picks the 32k vocabulary |
| `"smoke"` | any GPU / Apple GPU | the **entire pipeline on tiny settings** (a few minutes): data, pilot, pretraining, instruction tuning, export. Run this first, every time you change something |
| `"pilot"` | Kaggle **GPU T4 x2** | equal-wall-clock arms: distillation vs plain CE training, Muon vs AdamW. Decides what the long run uses |
| `"pretrain"` | Kaggle **GPU T4 x2** | the main run. Resumes from its checkpoint, stops cleanly before the 12 h limit. Run it again (new session, previous output attached) until it reports DONE; then it continues into instruction tuning |
| `"sft"` | Kaggle **GPU T4 x2** | instruction tuning (smol-smoltalk) with the instruct teacher, then export + report |

**The student** (51.5M parameters): 18 layers, d_model 512, 8 query heads sharing 2 key/value heads (**GQA**), per-head **QK-norm**, SwiGLU 1280, RoPE, RMSNorm, 512 context, 32,768-token vocabulary with a **factorized tied embedding** (32768 x 128, projected to 512). Deep and narrow, following MobileLLM. `STUDENT["block_repeats"] = 2` would run every block twice (MobileLLM-LS): double the depth for the same parameters.

**Why the run is built this way**
- The 0.5B teacher's forward pass is most of the compute, so a bigger student is almost free and **MoE would save compute where there is little**: the student stays dense.
- **Two T4s, one process each** (`torchrun`/DDP): each GPU holds its own teacher copy, gradients are averaged. Kaggle charges session hours, not GPU count, so this roughly doubles tokens per quota hour.
- **Muon** (orthogonalised momentum) for the transformer-block matrices, AdamW for embeddings and norms. A `USE_MUON` flag falls back to plain AdamW; the pilot measures whether it helps.
- **Warmup-stable-decay** schedule: the learning rate is flat until 80% of the run, so the run can be extended or stopped early without redoing the schedule.
- **Held-out sets**: Cosmopedia (the 10M run's validation text), held-out FineWeb-Edu, and **WikiText-103 (true out-of-domain: nothing trained on it)**. Everything is also reported in **bits per byte**, which is comparable to the 10M model despite its different vocabulary.
- Zero-shot **benchmarks** (ARC-Easy, HellaSwag, SciQ, LAMBADA) are scored during training, so you get accuracy *curves*.

**Dry run first.** `RUN_MODE = "smoke"`. It prints real tokens/s and memory, which replaces every estimate in these notes.

**Sessions.** Checkpoints (weights, optimizer, loss scaler, step, history) are written every `SAVE_EVERY` steps. To continue: new session -> attach the previous version's output (or a dataset holding `pretrain_latest.pt`) as an input -> Run All with the same `RUN_MODE`. Do not change `TOKENS_PER_STEP`, `BATCH_SIZE` or the model between sessions; `PRETRAIN_MAX_STEPS` may be changed freely until the decay phase starts (the last 20%).

In [ ]:
!pip install -q -U "transformers>=4.56" datasets accelerate safetensors pyarrow matplotlib

## Config

In [ ]:
import os, sys, time, glob, shutil, json
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

RUN_MODE = "smoke"       # "data" | "smoke" | "pilot" | "pretrain" | "sft"   (see the table above)
RUN_MODE = os.environ.get("DISTIL_MODE", RUN_MODE)   # lets the notebook run as a script
assert RUN_MODE in ("data", "smoke", "pilot", "pretrain", "sft"), RUN_MODE
SMOKE = RUN_MODE == "smoke"
AUTO_SFT = True          # "pretrain" mode: continue into instruction tuning + export once pretraining is DONE

IN_KAGGLE = os.path.exists("/kaggle/working")
RUN_NAME = "distil_hybrid50m" + ("_smoke" if SMOKE else "")
PERSIST_DIR = f"/kaggle/working/{RUN_NAME}" if IN_KAGGLE else os.path.abspath(f"{RUN_NAME}_run")
DATA_DIR = os.path.join(PERSIST_DIR, "data")
CKPT_DIR = os.path.join(PERSIST_DIR, "checkpoints")
VOCAB_PATH = os.path.join(PERSIST_DIR, "vocab.json")
CODE_DIR = os.path.abspath("hybrid50m_code")
CFG_PATH = os.path.join(PERSIST_DIR, "cfg.json")
for d in (DATA_DIR, CKPT_DIR, CODE_DIR):
    os.makedirs(d, exist_ok=True)

# ---------------- teacher + precision ----------------
TEACHER_NAME = "Qwen/Qwen2.5-0.5B"          # base model: we want its raw next-token distribution
SFT_TEACHER = "Qwen/Qwen2.5-0.5B-Instruct"
PRECISION = "fp16"       # "fp16" (T4), "bf16" (Ampere+ only), "fp32"

# ---------------- data (RUN_MODE "data") ----------------
COSMO_TOKENS = 650_000_000    # Cosmopedia v2: synthetic textbooks, stories, wikiHow, blog posts
WEB_TOKENS = 370_000_000      # FineWeb-Edu-dedup documents with educational score >= WEB_MIN_SCORE (about the top 41%)
WEB_MIN_SCORE = 3.0
COSMO_N_FILES = 24            # of 104 parquet files; evenly spaced row-groups inside each
WEB_N_FILES = 24              # of 234
VAL_TOKENS = 2_000_000        # per held-out set
BENCH_N = 1000                # examples per benchmark
VOCAB_SIZE = 32768
SFT_VOCAB_SHARE = 0.03        # the instruction data gets this share of vocab-selection weight

# ---------------- student (51.5M parameters) ----------------
STUDENT = dict(d_model=512, n_layers=18, n_heads=8, n_kv_heads=2, ffn_hidden=1280, max_seq_len=512,
               emb_dim=128, qk_norm=True, block_repeats=1)
SEQ_LEN = 512

# ---------------- stage 1: pretraining ----------------
KD = True                     # False: plain next-token CE, the teacher is never loaded (decide with the pilot)
USE_MUON = True               # False: AdamW everywhere
TOKENS_PER_STEP = 65_536      # global optimizer batch; per GPU it is BATCH_SIZE x SEQ_LEN x accumulation
BATCH_SIZE = 8                # micro-batch per GPU; lower it (4, 2) on out-of-memory - accumulation adapts
PRETRAIN_MAX_STEPS = 15_000   # x 65,536 = 983M tokens (~1 epoch of the mix)
SCHEDULE = "wsd"              # warmup-stable-decay
LR = 1.5e-3
WARMUP = 400
DECAY_FRAC = 0.2
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
ALPHA = 0.7                   # weight on KL(teacher || student); the rest goes to hard-label CE
TEMPERATURE = 1.0
REQUIRE_CHECKPOINT = False    # True: refuse to start from step 0 when pretrain_latest.pt is missing (use when continuing a run)
MAX_RUN_HOURS = 11.5          # Kaggle kills a session at 12 h: save and stop cleanly before that
LOG_EVERY = 25
EVAL_EVERY = 500
BENCH_EVERY = 2500            # 0 disables the benchmarks
SAVE_EVERY = 200
MILESTONE_EVERY = 2500        # also keep a model-only snapshot every N steps
VAL_WINDOWS = 256             # windows of SEQ_LEN tokens per held-out set
EVAL_BS = 8

# ---------------- pilot: equal-wall-clock comparison ----------------
PILOT_ARMS = [dict(name="kd_muon", kd=True, muon=True), dict(name="ce_muon", kd=False, muon=True), dict(name="kd_adamw", kd=True, muon=False)]
PILOT_MINUTES = 40            # per arm (constant learning rate, so the curves are directly comparable)
PILOT_EVAL_EVERY = 100

# ---------------- stage 2: instruction tuning (smol-smoltalk, first exchange of each conversation) ----------------
SFT_MAX_LEN = 448
SFT_MAX_EXAMPLES = 150_000
SFT_MAX_UNK_FRAC = 0.05
SFT_BATCH_SIZE = 8            # per GPU
SFT_GRAD_ACCUM = 4            # per GPU: 8 x 4 x 2 GPUs = 64 examples per step
SFT_EPOCHS = 2
SFT_LR = 5e-4
SFT_WARMUP = 50
SFT_ALPHA = 0.5               # 0: plain SFT without the teacher
SFT_EVAL_EVERY = 200
SFT_SAVE_EVERY = 100

SEED = 0
SAMPLE_PROMPTS = ["The water cycle is", "Photosynthesis is the process by which", "Once upon a time, there was a little",
                  "How to bake a loaf of bread:\n\nStep 1:", "In mathematics, a prime number is"]
SAMPLE_QUERIES = ["What is the capital of France?", "Give three tips for staying healthy.", "Explain what photosynthesis is.",
                  "Write a short story about a dog."]

if SMOKE:   # tiny end-to-end settings; the real architecture, so shapes and memory are the real ones
    COSMO_TOKENS, WEB_TOKENS, VAL_TOKENS, COSMO_N_FILES, WEB_N_FILES, BENCH_N = 700_000, 400_000, 150_000, 4, 4, 30
    TOKENS_PER_STEP, BATCH_SIZE, PRETRAIN_MAX_STEPS, WARMUP = 4096, 2, 24, 3
    LOG_EVERY, EVAL_EVERY, BENCH_EVERY, SAVE_EVERY, MILESTONE_EVERY, VAL_WINDOWS, EVAL_BS = 2, 12, 12, 8, 0, 16, 4
    PILOT_MINUTES, PILOT_EVAL_EVERY = 0.4, 4
    SFT_MAX_EXAMPLES, SFT_EPOCHS, SFT_BATCH_SIZE, SFT_GRAD_ACCUM, SFT_EVAL_EVERY, SFT_SAVE_EVERY, SFT_WARMUP = 600, 1, 4, 2, 4, 4, 3

STAGES = {"data": ["data"], "smoke": ["data", "pilot", "pretrain", "sft", "export"], "pilot": ["pilot"],
          "pretrain": ["pretrain"] + (["sft", "export"] if AUTO_SFT else []), "sft": ["sft", "export"]}[RUN_MODE]
PAUSED = False           # set when a stage stops at the session time limit: later stages are then skipped
NB_START = time.time()
KAGGLE_INPUT = "/kaggle/input"


def cfg_dict():
    out = {}
    for k, v in globals().items():
        if k.isupper() and not k.startswith("_"):
            try:
                json.dumps(v); out[k] = v
            except TypeError:
                pass
    return out


def restore_previous_run():
    # Kaggle wipes /kaggle/working between sessions. Attach the previous output (or datasets holding these files) as inputs.
    # Big read-only data files are linked in place; checkpoints are copied (they get rewritten). Existing files are never overwritten.
    if not IN_KAGGLE or SMOKE:   # smoke builds its own tiny data; linking the real files in would turn it into a full-size run
        return
    wanted = {n: (DATA_DIR, "link") for n in ("train.bin", "cosmo_val.bin", "web_val.bin", "wiki_val.bin", "token_counts.npy",
                                              "data_meta.json", "sft_raw.npz", "bench.json")}
    wanted.update({n: (PERSIST_DIR, "copy") for n in ("vocab.json",)})
    for n in ("pretrain_latest.pt", "pretrain_best.pt", "sft_latest.pt", "sft_best.pt"):
        wanted[n] = (CKPT_DIR, "copy")
    for pat in ("history_*.json", "state_*.json", "info_*.json", "pilot_*_latest.pt"):
        for hit in glob.glob(os.path.join(KAGGLE_INPUT, "**", pat), recursive=True):
            wanted.setdefault(os.path.basename(hit), (CKPT_DIR, "copy"))
    for name, (folder, how) in wanted.items():
        dst = os.path.join(folder, name)
        if os.path.lexists(dst):
            print(f"already present: {name}")
            continue
        hits = glob.glob(os.path.join(KAGGLE_INPUT, "**", name), recursive=True)
        if not hits:
            continue
        src = max(hits, key=os.path.getmtime)
        print(f"{'linking' if how == 'link' else 'restoring'} {name} <- {src} ({os.path.getsize(src) / 1e6:,.0f} MB)")
        (os.symlink if how == "link" else shutil.copyfile)(src, dst)


restore_previous_run()
accum_1gpu = TOKENS_PER_STEP // (BATCH_SIZE * SEQ_LEN)
print(("SMOKE TEST - " if SMOKE else "") + f"RUN_MODE={RUN_MODE} | stages {STAGES} | persisting to {PERSIST_DIR}")
print(f"pretraining: {PRETRAIN_MAX_STEPS:,} steps x {TOKENS_PER_STEP:,} tokens = {PRETRAIN_MAX_STEPS * TOKENS_PER_STEP / 1e6:,.0f}M tokens "
      f"({BATCH_SIZE} x {SEQ_LEN} per GPU, {accum_1gpu} accumulation steps on one GPU, {accum_1gpu // 2} on two)")

## Code

The modules are written to `hybrid50m_code/` and imported; `train.py` is what `torchrun` launches. `selftest.py` checks the pieces everything depends on (below).

In [ ]:
%%writefile hybrid50m_code/model.py
"""~50M-parameter student: deep-and-narrow Llama-style decoder (MobileLLM recipe).

GQA (few key/value heads), QK-norm (fp16 stability), RoPE, RMSNorm, SwiGLU and a factorized tied embedding.
Optional `block_repeats` runs every stored block several times in a row (MobileLLM-LS): more depth, same parameters.
Also loads the weights of the earlier 10M model (fused qkv, no QK-norm) through `from_v4_state_dict`.
"""
import math
from dataclasses import dataclass, asdict, fields
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class Config:
    vocab_size: int = 32768
    d_model: int = 512
    n_layers: int = 18
    n_heads: int = 8
    n_kv_heads: int = 2
    ffn_hidden: int = 1280
    max_seq_len: int = 512
    rope_theta: float = 10000.0
    dropout: float = 0.0
    emb_dim: Optional[int] = 128   # factorized tied embedding: vocab * emb_dim, projected up/down (None: plain tied)
    qk_norm: bool = True
    block_repeats: int = 1         # >1: weight sharing across depth (compute grows, parameters do not)

    def to_dict(self):
        return asdict(self)

    @classmethod
    def from_dict(cls, d):
        known = {f.name for f in fields(cls)}
        return cls(**{k: v for k, v in d.items() if k in known})

    @classmethod
    def from_v4(cls, d):
        # the 10M model: plain multi-head attention, no QK-norm
        return cls.from_dict({**d, "n_kv_heads": d["n_heads"], "qk_norm": False, "block_repeats": 1})


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()  # statistics in fp32 so fp16 activations cannot overflow in the square
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).to(x.dtype) * self.weight


def rope_cache(seq_len, head_dim, theta, device):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    freqs = torch.outer(torch.arange(seq_len, device=device).float(), inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    # x: (B, H, T, D); rotate interleaved pairs in fp32, hand back x's dtype
    xf = x.float()
    x1, x2 = xf[..., 0::2], xf[..., 1::2]
    cos, sin = cos[None, None], sin[None, None]
    return torch.stack((x1 * cos - x2 * sin, x1 * sin + x2 * cos), dim=-1).flatten(-2).to(x.dtype)


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0 and cfg.n_heads % cfg.n_kv_heads == 0
        self.h, self.kv, self.hd = cfg.n_heads, cfg.n_kv_heads, cfg.d_model // cfg.n_heads
        self.wq = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.out = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)
        self.q_norm = RMSNorm(self.hd) if cfg.qk_norm else nn.Identity()
        self.k_norm = RMSNorm(self.hd) if cfg.qk_norm else nn.Identity()
        self.dropout = cfg.dropout

    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.wk(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.wv(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        q, k = apply_rope(self.q_norm(q), cos, sin), apply_rope(self.k_norm(k), cos, sin)
        if self.kv != self.h:  # share each key/value head between h/kv query heads
            k, v = k.repeat_interleave(self.h // self.kv, dim=1), v.repeat_interleave(self.h // self.kv, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        return self.out(y.transpose(1, 2).reshape(B, T, self.h * self.hd))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = Attention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.mlp = SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin):
        x = x + self.drop(self.attn(self.norm1(x), cos, sin))
        return x + self.drop(self.mlp(self.norm2(x)))


class StudentLM(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        e = cfg.emb_dim or cfg.d_model
        self.embed = nn.Embedding(cfg.vocab_size, e)
        self.proj_in = nn.Linear(e, cfg.d_model, bias=False) if cfg.emb_dim else nn.Identity()
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model)
        self.proj_out = nn.Linear(cfg.d_model, e, bias=False) if cfg.emb_dim else nn.Identity()
        self.lm_head = nn.Linear(e, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight  # tied

        cos, sin = rope_cache(cfg.max_seq_len, cfg.d_model // cfg.n_heads, cfg.rope_theta, "cpu")
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        if cfg.emb_dim:  # narrow embeddings need a larger std so the projected signal is not tiny
            nn.init.normal_(self.embed.weight, mean=0.0, std=0.1)
        depth = cfg.n_layers * cfg.block_repeats
        for name, p in self.named_parameters():  # GPT-2 style scaled init for the residual-branch outputs
            if name.endswith("attn.out.weight") or name.endswith("mlp.down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * depth))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def forward(self, idx):
        T = idx.shape[1]
        assert T <= self.cfg.max_seq_len, f"sequence length {T} > max_seq_len {self.cfg.max_seq_len}"
        x = self.proj_in(self.embed(idx))
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        for block in self.blocks:
            for _ in range(self.cfg.block_repeats):
                x = block(x, cos, sin)
        return self.lm_head(self.proj_out(self.norm(x)))

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50, stop_id=None, repetition_penalty=1.0):
        # no KV cache on purpose: this is for samples during training. Inference gets its own cached path at deployment.
        for _ in range(max_new_tokens):
            logits = self(idx[:, -self.cfg.max_seq_len:])[:, -1, :].float()
            if repetition_penalty != 1.0:
                seen = torch.gather(logits, 1, idx[:, -256:])
                logits.scatter_(1, idx[:, -256:], torch.where(seen > 0, seen / repetition_penalty, seen * repetition_penalty))
            if temperature <= 0:
                nxt = logits.argmax(-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float("inf")
                nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
            idx = torch.cat([idx, nxt], dim=1)
            if stop_id is not None and (nxt == stop_id).all():
                break
        return idx


def from_v4_state_dict(sd, cfg: Config):
    """Map a 10M-model state dict (fused `qkv`) onto this architecture (needs n_kv_heads == n_heads, qk_norm False)."""
    d = cfg.d_model
    out = {}
    for k, v in sd.items():
        if k == "lm_head.weight":
            continue
        if k.endswith("attn.qkv.weight"):
            pre = k[: -len("qkv.weight")]
            out[pre + "wq.weight"], out[pre + "wk.weight"], out[pre + "wv.weight"] = v[:d], v[d:2 * d], v[2 * d:]
        else:
            out[k] = v
    return out

In [ ]:
%%writefile hybrid50m_code/optim.py
"""Optimizer + learning-rate schedules.

`MuonAdamW`: Muon (orthogonalised momentum) for the 2-D matrices inside the transformer blocks, AdamW for everything else
(embeddings, projections, norms). One torch Optimizer, so GradScaler / state_dict / checkpoints treat it like any other.
Muon's update is rescaled to the RMS of an AdamW update (Moonlight, 0.2*sqrt(max(shape))), so ONE learning rate and ONE
weight decay serve both halves. `muon=False` gives plain AdamW with the same parameter groups.
"""
import math

import torch


def newton_schulz(G, steps=5):
    # quintic iteration that pushes all singular values of G towards 1 (the "orthogonalised" update); fp32 on purpose
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float()
    flip = X.size(0) > X.size(1)
    if flip:
        X = X.T
    X = X / (X.norm() + 1e-7)
    for _ in range(steps):
        A = X @ X.T
        X = a * X + (b * A + c * (A @ A)) @ X
    return X.T if flip else X


class MuonAdamW(torch.optim.Optimizer):
    def __init__(self, groups, lr, betas=(0.9, 0.95), eps=1e-8, momentum=0.95, ns_steps=5):
        # groups: list of dicts {"params": [...], "kind": "muon" | "adamw", "weight_decay": float}
        defaults = dict(lr=lr, betas=betas, eps=eps, momentum=momentum, ns_steps=ns_steps, weight_decay=0.0, kind="adamw")
        super().__init__(groups, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for g in self.param_groups:
            lr, wd = g["lr"], g["weight_decay"]
            for p in g["params"]:
                if p.grad is None:
                    continue
                st = self.state[p]
                if g["kind"] == "muon":
                    if "buf" not in st:
                        st["buf"] = torch.zeros_like(p)
                    st["buf"].mul_(g["momentum"]).add_(p.grad)
                    upd = newton_schulz(p.grad + g["momentum"] * st["buf"], g["ns_steps"])   # Nesterov momentum
                    p.mul_(1 - lr * wd)
                    p.add_(upd.to(p.dtype), alpha=-lr * 0.2 * math.sqrt(max(p.shape)))
                else:
                    if not st:
                        st["t"] = 0
                        st["m"], st["v"] = torch.zeros_like(p), torch.zeros_like(p)
                    st["t"] += 1
                    b1, b2 = g["betas"]
                    st["m"].mul_(b1).add_(p.grad, alpha=1 - b1)
                    st["v"].mul_(b2).addcmul_(p.grad, p.grad, value=1 - b2)
                    denom = (st["v"] / (1 - b2 ** st["t"])).sqrt_().add_(g["eps"])
                    p.mul_(1 - lr * wd)
                    p.addcdiv_(st["m"], denom, value=-lr / (1 - b1 ** st["t"]))


def build_optimizer(model, lr, weight_decay=0.1, muon=True):
    muon_p, decay_p, plain_p = [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if muon and p.dim() == 2 and name.startswith("blocks."):
            muon_p.append(p)
        elif p.dim() >= 2:
            decay_p.append(p)
        else:
            plain_p.append(p)
    groups = [{"params": decay_p, "kind": "adamw", "weight_decay": weight_decay},
              {"params": plain_p, "kind": "adamw", "weight_decay": 0.0}]
    if muon_p:
        groups.insert(0, {"params": muon_p, "kind": "muon", "weight_decay": weight_decay})
    return MuonAdamW(groups, lr=lr)


def lr_at(step, total, base, warmup, kind="wsd", decay_frac=0.2, min_ratio=0.02):
    """Learning rate for optimizer step `step` (0-based).

    wsd      warmup -> flat -> last `decay_frac` of the run decays as 1 - sqrt(progress). The flat part does not depend on
             `total`, so a run can be extended (or stopped) without redoing the schedule, as long as it has not started decaying.
    cosine   warmup -> cosine to min_ratio * base
    constant warmup -> flat (used by the pilot, where arms are compared at equal wall-clock time)
    """
    if step < warmup:
        return base * (step + 1) / warmup
    if kind == "constant":
        return base
    if kind == "cosine":
        p = min(1.0, (step - warmup) / max(1, total - warmup))
        return base * (min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * p)))
    start = int(total * (1 - decay_frac))
    if step < start:
        return base
    p = min(1.0, (step - start) / max(1, total - start))
    return base * (min_ratio + (1 - min_ratio) * (1 - math.sqrt(p)))

In [ ]:
%%writefile hybrid50m_code/core.py
"""Shared pieces for the 50M distillation run: vocab map, teacher, KD loss, token files, evaluation, benchmarks,
checkpoints, export and the dashboard figure. Used by train.py and by the notebook."""
import json
import math
import os
import time

import numpy as np
import torch
import torch.nn.functional as F

from model import Config, StudentLM

NEG_LOGIT = -1e4   # stands in for -inf so 0 * (t - s) stays 0 inside the KL
PROMPT_TEMPLATE = "### Question:\n{query}\n\n### Answer:\n"
LN2 = math.log(2)


def hms(seconds):
    seconds = int(max(0, seconds))
    return f"{seconds // 3600}:{seconds % 3600 // 60:02d}:{seconds % 60:02d}"


# ------------------------------------------------------------------ vocabulary
class VocabMap:
    """Student ids 0..K-1 are a subset of the teacher's ids; student id K is UNK (every other teacher token)."""

    def __init__(self, data):
        if not isinstance(data, dict):
            data = json.load(open(data))
        self.raw = data
        self.teacher_ids = torch.tensor(data["teacher_ids"], dtype=torch.long)
        self.unk_id = len(self.teacher_ids)
        self.size = self.unk_id + 1
        lookup = torch.full((data["teacher_vocab_size"],), self.unk_id, dtype=torch.long)
        lookup[self.teacher_ids] = torch.arange(len(self.teacher_ids))
        self.t2s = lookup
        self.eos_id = int(lookup[data["eos_teacher_id"]])
        self._dev = {}
        self.sbytes = None

    def _on(self, device):
        if device not in self._dev:
            self._dev[device] = self.t2s.to(device)
        return self._dev[device]

    def to_student(self, ids):
        if isinstance(ids, torch.Tensor):
            return self._on(ids.device)[ids]
        return self.t2s[torch.tensor(ids, dtype=torch.long)].tolist()

    def to_teacher(self, ids):  # for decoding; UNK has no teacher token and is dropped
        return [int(self.teacher_ids[i]) for i in ids if i != self.unk_id]

    def set_bytes(self, tok_bytes):
        # UTF-8 bytes each student token stands for (UNK: 0). Bits-per-byte is comparable across tokenizers, CE is not.
        b = torch.tensor(np.asarray(tok_bytes), dtype=torch.float32)[self.teacher_ids]
        self.sbytes = torch.cat([b, torch.zeros(1)])


def teacher_token_bytes(tokenizer, teacher_vocab):
    """Byte length of every teacher token. Qwen's tokens are byte-level BPE: one character of the token string = one byte."""
    out = np.zeros(teacher_vocab, np.float32)
    for tok, i in tokenizer.get_vocab().items():
        if i < teacher_vocab:
            out[i] = len(tok)
    for i in tokenizer.added_tokens_decoder:      # <|endoftext|> and friends stand for no text
        if i < teacher_vocab:
            out[i] = 0
    return out


# ------------------------------------------------------------------ teacher + loss
class Teacher:
    """Frozen teacher whose logits are computed only over the student's vocabulary subset (hidden states x K head rows)."""

    def __init__(self, name, vocab, device, dtype):
        from transformers import AutoModelForCausalLM
        model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype)
        model.to(device).eval().requires_grad_(False)
        self.name, self.dtype = name, dtype
        self.backbone = model.get_decoder()
        self.head = model.get_output_embeddings().weight.index_select(0, vocab.teacher_ids.to(device)).contiguous()
        del model

    @torch.no_grad()
    def logits(self, input_ids):
        h = self.backbone(input_ids=input_ids).last_hidden_state
        logits = (h @ self.head.T).float()
        unk = torch.full((*logits.shape[:-1], 1), NEG_LOGIT, device=logits.device)
        return torch.cat([logits, unk], dim=-1)


def kd_loss(student_logits, teacher_logits, targets, temperature=1.0, alpha=0.5):
    """alpha * T^2 * KL(teacher || student) + (1 - alpha) * CE over targets != -100. Returns (total, kl, ce)."""
    V = student_logits.size(-1)
    s = student_logits.float().view(-1, V)
    tgt = targets.reshape(-1)
    n = (tgt != -100).float().sum().clamp(min=1)
    ce = F.cross_entropy(s, tgt, ignore_index=-100, reduction="sum") / n
    if teacher_logits is None or alpha == 0:
        return ce, torch.zeros((), device=ce.device), ce
    t_logp = F.log_softmax(teacher_logits.view(-1, V) / temperature, dim=-1)
    s_logp = F.log_softmax(s / temperature, dim=-1)
    kl = ((t_logp.exp() * (t_logp - s_logp)).sum(-1) * (tgt != -100).float()).sum() / n * temperature ** 2
    return alpha * kl + (1 - alpha) * ce, kl, ce


def split_batch(batch, vocab, device):
    # batch: (B, T+1) teacher ids -> teacher input, student input, student targets (UNK targets ignored)
    batch = batch.to(device, non_blocking=True)
    xt = batch[:, :-1]
    y = vocab.to_student(batch[:, 1:]).clone()
    y[y == vocab.unk_id] = -100
    return xt, vocab.to_student(xt), y


def load_teacher_checked(name, vocab, device, dtype, probe_batches, log=print):
    """fp16 teacher, verified against fp32 on a few batches; falls back to fp32 if fp16 misbehaves."""
    def ce_of(t):
        tot = 0.0
        for b in probe_batches:
            xt, _, y = split_batch(b, vocab, device)
            tot += kd_loss(t.logits(xt), None, y, alpha=0.0)[2].item()
        return tot / len(probe_batches)

    t = Teacher(name, vocab, device, dtype)
    if dtype == torch.float32 or not probe_batches:
        return t
    finite = all(bool(torch.isfinite(t.logits(split_batch(b, vocab, device)[0])[..., :-1]).all()) for b in probe_batches)
    ce_low = ce_of(t)
    ref = Teacher(name, vocab, device, torch.float32)
    ce_ref = ce_of(ref)
    ok = finite and abs(ce_low - ce_ref) < 0.02
    log(f"teacher {str(dtype).replace('torch.', '')} check: CE {ce_low:.4f} vs fp32 {ce_ref:.4f} (finite: {finite}) -> "
        + ("OK" if ok else "MISMATCH, using fp32 teacher"))
    if ok:
        del ref
        return t
    del t
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return ref


# ------------------------------------------------------------------ token files
class TokenFile:
    """Flat uint32 token file read through np.memmap as consecutive windows of seq_len + 1 tokens."""

    def __init__(self, path, seq_len):
        self.path, self.seq_len = path, seq_len
        self.n_tokens = os.path.getsize(path) // 4
        self.n_windows = max(0, (self.n_tokens - 1) // seq_len)
        self.mm = None

    def get(self, idx):
        if self.mm is None:
            self.mm = np.memmap(self.path, dtype=np.uint32, mode="r")
        T = self.seq_len
        return torch.from_numpy(np.stack([self.mm[i * T: i * T + T + 1] for i in idx]).astype(np.int64))

    def fixed(self, n_windows):
        # n evenly spaced windows: the same fixed evaluation batch every time, representative of the whole file
        return self.get(np.linspace(0, self.n_windows - 1, min(n_windows, self.n_windows)).astype(int))


class WindowSampler:
    """Infinite shuffled stream of window ids. `take(start, count)` is a pure function of its arguments (epoch permutations
    are seeded), so a resumed run continues with exactly the next windows."""

    def __init__(self, n, seed):
        self.n, self.seed, self._ep, self._perm = n, seed, -1, None

    def take(self, start, count):
        out = []
        while count:
            ep, off = divmod(start, self.n)
            if ep != self._ep:
                self._perm = torch.randperm(self.n, generator=torch.Generator().manual_seed(self.seed + ep)).numpy()
                self._ep = ep
            k = min(count, self.n - off)
            out.append(self._perm[off: off + k])
            start, count = start + k, count - k
        return np.concatenate(out)


# ------------------------------------------------------------------ evaluation
@torch.no_grad()
def evaluate(model, vocab, windows, device, amp, teacher=None, batch_size=8):
    """Held-out metrics over fixed windows. ce: nats per predicted token (UNK targets excluded); bpb: bits per UTF-8 byte of
    those tokens; with a teacher also kl (teacher||student), top-1 agreement and the teacher's own ce / bpb."""
    was_training = model.training
    model.eval()
    nll = byt = n = kl_sum = agree = t_nll = 0.0
    for i in range(0, len(windows), batch_size):
        xt, xs, y = split_batch(windows[i: i + batch_size], vocab, device)
        with amp():
            s = model(xs)
        s = s.float()
        V = s.size(-1)
        valid = y != -100
        yy = y.clamp(min=0)
        s_logp = F.log_softmax(s, dim=-1)
        nll += -(s_logp.gather(-1, yy[..., None]).squeeze(-1) * valid).sum().item()
        byt += (vocab.sbytes.to(device)[yy] * valid).sum().item()
        n += valid.sum().item()
        if teacher is not None:
            t_logp = F.log_softmax(teacher.logits(xt), dim=-1)
            kl_sum += ((t_logp.exp() * (t_logp - s_logp)).sum(-1) * valid).sum().item()
            agree += ((s.argmax(-1) == t_logp.argmax(-1)) & valid).sum().item()
            t_nll += -(t_logp.gather(-1, yy[..., None]).squeeze(-1) * valid).sum().item()
    model.train(was_training)
    n = max(n, 1)
    out = {"ce": nll / n, "ppl": math.exp(min(20, nll / n)), "bpb": nll / LN2 / max(byt, 1)}
    if teacher is not None:
        out.update(kl=kl_sum / n, agree=100 * agree / n, t_ce=t_nll / n, t_bpb=t_nll / LN2 / max(byt, 1))
    return out


# ------------------------------------------------------------------ benchmarks (zero-shot, log-likelihood)
def prep_bench(raw, tokenizer):
    """bench.json (plain text) -> token ids, tokenized once."""
    enc = lambda s: tokenizer(s)["input_ids"]
    out = {}
    for name, task in raw.items():
        items = []
        for it in task["items"]:
            if task["type"] == "mc":
                items.append(dict(ctx=enc(it["ctx"]), conts=[enc(c) for c in it["choices"]],
                                  nbytes=[max(1, len(c.encode())) for c in it["choices"]], label=it["label"]))
            else:
                items.append(dict(ctx=enc(it["ctx"]), conts=[enc(it["target"])]))
        out[name] = dict(type=task["type"], items=items)
    return out


@torch.no_grad()
def _score(model, vocab, seqs, device, amp, max_len, bs=32):
    # seqs: [(teacher ids, n_cont)] -> per sequence (sum log-prob of non-UNK continuation tokens, #scored, all-greedy-correct, has_unk)
    model.eval()
    res = [None] * len(seqs)
    order = sorted(range(len(seqs)), key=lambda i: len(seqs[i][0]))
    for k in range(0, len(order), bs):
        rows = order[k: k + bs]
        items = []
        for i in rows:
            ids, c = seqs[i]
            ids = ids[-(max_len + 1):]
            items.append((vocab.to_student(ids), min(c, len(ids) - 1)))
        L = max(len(s) for s, _ in items)
        x = torch.zeros(len(rows), L, dtype=torch.long)
        cmask = torch.zeros(len(rows), L - 1, dtype=torch.bool)
        for r, (s, c) in enumerate(items):
            x[r, : len(s)] = torch.tensor(s)
            cmask[r, len(s) - c - 1: len(s) - 1] = True
        x, cmask = x.to(device), cmask.to(device)
        with amp():
            logits = model(x[:, :-1])
        logp = F.log_softmax(logits.float(), dim=-1)
        tgt = x[:, 1:]
        known = tgt != vocab.unk_id
        lp = logp.gather(-1, tgt[..., None]).squeeze(-1)
        use = cmask & known
        ok = ((logp.argmax(-1) == tgt) | ~cmask).all(1)
        has_unk = (cmask & ~known).any(1)
        for r, i in enumerate(rows):
            res[i] = ((lp[r] * use[r]).sum().item(), int(use[r].sum()), bool(ok[r]), bool(has_unk[r]))
    return res


def run_bench(model, vocab, prepared, device, amp, max_len):
    """Zero-shot scores: acc / acc_norm (norm = by byte length of the choice) for multiple choice, accuracy + perplexity of the last word for LAMBADA."""
    out = {}
    for name, task in prepared.items():
        items = task["items"]
        if task["type"] == "mc":
            seqs = [(it["ctx"] + c, len(c)) for it in items for c in it["conts"]]
            res = _score(model, vocab, seqs, device, amp, max_len)
            hit = hit_n = 0
            pos = 0
            for it in items:
                k = len(it["conts"])
                sc = np.array([r[0] for r in res[pos: pos + k]])
                nb = np.array(it["nbytes"], dtype=float)
                hit += int(sc.argmax() == it["label"])
                hit_n += int((sc / nb).argmax() == it["label"])
                pos += k
            out[f"{name}"] = 100 * hit / len(items)
            out[f"{name}_norm"] = 100 * hit_n / len(items)
        else:
            res = _score(model, vocab, [(it["ctx"] + it["conts"][0], len(it["conts"][0])) for it in items], device, amp, max_len)
            good = [r for r in res if not r[3]]
            out[f"{name}_acc"] = 100 * sum(r[2] for r in good) / max(1, len(good))
            out[f"{name}_ppl"] = math.exp(-sum(r[0] for r in good) / max(1, len(good)))
    return out


# ------------------------------------------------------------------ checkpoints, sampling, export
def save_checkpoint(path, model, optimizer, step, vocab_json, scaler=None, **extra):
    tmp = path + ".tmp"   # write-then-rename: a disconnect mid-save cannot corrupt the previous checkpoint
    torch.save({"model": model.state_dict(), "config": model.cfg.to_dict(), "vocab": vocab_json,
                "optimizer": optimizer.state_dict() if optimizer is not None else None,
                "scaler": scaler.state_dict() if scaler is not None else None, "step": step, **extra}, tmp)
    os.replace(tmp, path)


def load_checkpoint(path, device, vocab_json=None):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    if vocab_json is not None and "vocab" in ckpt:
        assert ckpt["vocab"]["teacher_ids"] == vocab_json["teacher_ids"], f"{path} was trained with a DIFFERENT vocab"
    model = StudentLM(Config.from_dict(ckpt["config"]))
    model.load_state_dict(ckpt["model"])
    return model.to(device), ckpt


@torch.no_grad()
def sample_text(model, vocab, tokenizer, prompt, device, amp, max_new_tokens=80, temperature=0.8, top_k=50,
                repetition_penalty=1.0):
    was_training = model.training
    model.eval()
    ids = vocab.to_student(tokenizer(prompt)["input_ids"])
    with amp():
        out = model.generate(torch.tensor([ids], device=device), max_new_tokens, temperature, top_k, stop_id=vocab.eos_id,
                             repetition_penalty=repetition_penalty)[0, len(ids):].tolist()
    if vocab.eos_id in out:
        out = out[: out.index(vocab.eos_id)]
    model.train(was_training)
    return tokenizer.decode(vocab.to_teacher(out))


def export_final(ckpt_path, out_path):
    from safetensors.torch import save_file
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    tensors = {k: v.half().contiguous() for k, v in ck["model"].items() if k != "lm_head.weight"}   # tied to embed.weight
    meta = {"format": "distil-student-v2", "config": json.dumps(ck["config"]), "vocab": json.dumps(ck["vocab"]),
            "step": str(ck["step"]), "prompt_template": PROMPT_TEMPLATE}
    save_file(tensors, out_path, metadata=meta)
    print(f"saved {out_path} ({os.path.getsize(out_path) / 1e6:.1f} MB)")


def read_json(path, default=None):
    try:
        return json.load(open(path))
    except Exception:
        return default


# ------------------------------------------------------------------ history + dashboard
class History:
    """Train / eval / bench records of one run; persisted so plots continue across resumed sessions."""

    def __init__(self, path, start_step=0):
        h = read_json(path, {})
        # a resumed run replays from its last checkpoint: drop anything recorded after it
        self.h = {k: [r for r in h.get(k, []) if r["step"] <= start_step] for k in ("train", "eval", "bench")}
        self.path = path

    def add(self, kind, rec):
        self.h[kind].append({k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in rec.items()})

    def save(self):
        json.dump(self.h, open(self.path + ".tmp", "w"))
        os.replace(self.path + ".tmp", self.path)


def plot_history(h, png, title=""):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    tr, ev, be = h["train"], h["eval"], h["bench"]
    if len(tr) < 2:
        return False
    col = lambda rs, k: np.array([r.get(k, np.nan) for r in rs], dtype=float)
    fig, ax = plt.subplots(2, 3, figsize=(17, 8))
    s = col(tr, "step")
    for k in ("loss", "ce", "kl"):
        if not np.all(np.isnan(col(tr, k))) and np.nanmax(col(tr, k)) > 0:
            ax[0, 0].plot(s, col(tr, k), label=k)
    ax[0, 0].set_title("training loss"); ax[0, 0].legend()
    if ev:
        t = col(ev, "tokens") / 1e6
        unit = "bpb" if any(k.endswith("/bpb") for r in ev for k in r) else "ce"
        for k in sorted({k for r in ev for k in r if k.endswith("/" + unit)}):
            ax[0, 1].plot(t, col(ev, k), marker="o", label=k[: -len(unit) - 1])
        for k in sorted({k for r in ev for k in r if k.endswith("/t_" + unit)}):
            ax[0, 1].axhline(np.nanmean(col(ev, k)), ls=":", c="gray", lw=1)
        ax[0, 1].set_title(f"held-out {'bits/byte' if unit == 'bpb' else 'CE'} (dotted: teacher)"); ax[0, 1].set_xlabel("M tokens"); ax[0, 1].legend()
        for k in ("cosmo/kl", "cosmo/agree"):
            if any(k in r for r in ev):
                ax[0, 2].plot(t, col(ev, k), marker="o", label=k)
        ax[0, 2].set_title("agreement with the teacher"); ax[0, 2].set_xlabel("M tokens")
        if ax[0, 2].get_legend_handles_labels()[0]:
            ax[0, 2].legend()
    if be:
        t = col(be, "tokens") / 1e6
        for k in sorted({k for r in be for k in r if k not in ("step", "tokens", "t") and not k.endswith("_ppl")}):
            ax[1, 0].plot(t, col(be, k), marker="o", label=k)
        ax[1, 0].set_title("zero-shot benchmarks (%)"); ax[1, 0].set_xlabel("M tokens"); ax[1, 0].legend(fontsize=7)
    ax[1, 1].plot(s, col(tr, "lr"), label="lr"); ax[1, 1].set_title("learning rate / grad norm")
    ax2 = ax[1, 1].twinx(); ax2.plot(s, col(tr, "gnorm"), c="tab:red", alpha=0.6)
    ax[1, 2].plot(s, col(tr, "tok_s") / 1e3); ax[1, 2].set_title("throughput (k tokens/s, all GPUs)")
    for a in ax.flat:
        a.grid(alpha=0.3)
    ax[0, 0].set_xlabel("step"); ax[1, 1].set_xlabel("step"); ax[1, 2].set_xlabel("step")
    fig.suptitle(f"{title}  step {int(s[-1])}")
    fig.tight_layout()
    fig.savefig(png, dpi=100)
    plt.close(fig)
    return True

In [ ]:
%%writefile hybrid50m_code/data.py
"""Data stage (needs no GPU): download + tokenize the training mix and held-out sets, pick the student vocabulary,
tokenize the instruction data, download the benchmark sets. Everything lands in C.DATA_DIR as plain files."""
import itertools
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np

from core import PROMPT_TEMPLATE, hms

SMOLLM = "datasets/HuggingFaceTB/smollm-corpus"
COSMO_FILES, WEB_FILES = 104, 234
COSMO_VAL_REF = dict(n_files=16, ref_rounds=44, tokens_per_doc=700)   # the 10M run's validation recipe, kept so val text stays the same


def paths(data_dir):
    p = lambda n: os.path.join(data_dir, n)
    return dict(train=p("train.bin"), cosmo_val=p("cosmo_val.bin"), web_val=p("web_val.bin"), wiki_val=p("wiki_val.bin"),
                counts=p("token_counts.npy"), meta=p("data_meta.json"), sft=p("sft_raw.npz"), bench=p("bench.json"))


def retry(fn, tries=8):
    # the Hub rate-limits parallel readers (HTTP 429): back off and try again
    for k in range(tries):
        try:
            return fn()
        except Exception:
            if k == tries - 1:
                raise
            time.sleep(2 ** k)


def open_files(subdir, total, n_files):
    import pyarrow.parquet as pq
    from huggingface_hub import HfFileSystem
    fs = HfFileSystem()
    idx = np.linspace(0, total - 1, n_files).astype(int)
    path = lambda i: f"{SMOLLM}/{subdir}/train-{i:05d}-of-{total:05d}.parquet"
    with ThreadPoolExecutor(n_files) as ex:
        pfs = list(ex.map(lambda i: retry(lambda: pq.ParquetFile(fs.open(path(i), "rb"))), idx))
    return idx.tolist(), pfs


def read_group(pf, gi, columns):
    # one unreadable row-group costs nothing (the files are shuffled): skip it instead of losing a half-finished tokenization
    try:
        return retry(lambda: pf.read_row_group(gi, columns=columns))
    except Exception as e:
        print(f"skipping row-group {gi} ({type(e).__name__}: {e})")
        return None


def read_cosmo(pf, gi):
    t = read_group(pf, gi, ["text"])
    return t.column("text").to_pylist() if t is not None else []


_WEB_NESTED_OK = [True]


def read_web(pf, gi, min_score, min_tokens=100):
    # FineWeb-Edu-dedup rows carry the classifier's educational score: keep only the better-than-`min_score` ones
    if _WEB_NESTED_OK[0]:
        t = read_group(pf, gi, ["text", "metadata.score", "metadata.token_count"])
        if t is not None:
            try:
                meta = t.column("metadata").to_pylist()
                return [x for x, m in zip(t.column("text").to_pylist(), meta)
                        if m["score"] >= min_score and m["token_count"] >= min_tokens]
            except Exception as e:
                print(f"metadata filter unavailable ({type(e).__name__}); keeping unfiltered text")
                _WEB_NESTED_OK[0] = False
    t = read_group(pf, gi, ["text"])
    return t.column("text").to_pylist() if t is not None else []


def rounds_iter(pfs, groups, read_fn):
    # one row-group per file per round, read by one thread per file; the next round downloads while this one is tokenized
    nf, rounds = len(pfs), min(len(g) for g in groups)
    with ThreadPoolExecutor(nf) as pool:
        submit = lambda r: [pool.submit(read_fn, pfs[i], groups[i][r]) for i in range(nf)]
        cur = submit(0)
        for r in range(rounds):
            nxt = submit(r + 1) if r + 1 < rounds else None
            yield [t for f in cur for t in f.result()]
            cur = nxt


def spread(n_groups, rounds, exclude=()):
    g = np.linspace(0, n_groups - 2, rounds).astype(int)
    return [int(x) for x in dict.fromkeys(g.tolist()) if x not in exclude]


def tokenize_into(f, texts, tok, eos, counts=None):
    n = nd = 0
    for i in range(0, len(texts), 256):
        enc = tok(texts[i: i + 256])["input_ids"]
        flat = np.concatenate([np.asarray(x + [eos], dtype=np.uint32) for x in enc])
        flat.tofile(f)
        if counts is not None:
            counts += np.bincount(flat, minlength=len(counts))
        n += len(flat); nd += len(enc)
    return n, nd


def wikitext_test():
    # a corpus neither the student nor the teacher's data selection ever saw: true out-of-domain check (approximate detokenization)
    import pyarrow.parquet as pq
    from huggingface_hub import HfFileSystem
    pf = retry(lambda: pq.ParquetFile(HfFileSystem().open(
        "datasets/Salesforce/wikitext/wikitext-103-raw-v1/test-00000-of-00001.parquet", "rb")))
    s = "".join(retry(lambda: pf.read()).column("text").to_pylist())
    for a, b in ((" @-@ ", "-"), (" @,@ ", ","), (" @.@ ", "."), (" , ", ", "), (" . ", ". "), (" ; ", "; "), (" : ", ": "),
                 (" 's", "'s"), (" n't", "n't"), ("( ", "("), (" )", ")"), (" \n", "\n")):
        s = s.replace(a, b)
    return re.sub(r"\n{3,}", "\n\n", s)


def prepare_training_data(C, tok, eos, teacher_vocab):
    P = paths(C.DATA_DIR)
    need = [P[k] for k in ("train", "cosmo_val", "web_val", "counts", "meta")]
    if all(os.path.exists(p) for p in need):
        print("token files already on disk:", json.load(open(P["meta"])))
        return
    t0 = time.time()
    counts = np.zeros(teacher_vocab, np.int64)
    meta = dict(tokens={}, docs={})

    def write(name, texts, count=False, f=None):
        own = f is None
        f = f or open(P[name] + ".tmp", "wb")
        n, nd = tokenize_into(f, texts, tok, eos, counts if count else None)
        if own:
            f.close(); os.replace(P[name] + ".tmp", P[name])
        meta["tokens"][name] = meta["tokens"].get(name, 0) + n
        meta["docs"][name] = meta["docs"].get(name, 0) + nd
        return n

    # ---- validation sets (never read by training) ----
    ref = COSMO_VAL_REF
    ref_idx, ref_pfs = open_files("cosmopedia-v2", COSMO_FILES, ref["n_files"])
    val_groups = {fi: min(max(1, (pf.metadata.num_row_groups // ref["ref_rounds"]) // 2), pf.metadata.num_row_groups - 1)
                  for fi, pf in zip(ref_idx, ref_pfs)}
    per_file = int(np.ceil(C.VAL_TOKENS / (ref["n_files"] * ref["tokens_per_doc"])))
    with ThreadPoolExecutor(len(ref_pfs)) as ex:
        texts = [t for ts in ex.map(lambda a: read_cosmo(a[1], val_groups[a[0]])[:per_file], zip(ref_idx, ref_pfs)) for t in ts]
    write("cosmo_val", texts)

    web_idx, web_pfs = open_files("fineweb-edu-dedup", WEB_FILES, C.WEB_N_FILES)
    web_val_groups = {fi: pf.metadata.num_row_groups // 2 for fi, pf in zip(web_idx, web_pfs)}
    n_val_files = min(len(web_pfs), 16)
    per_file = int(np.ceil(C.VAL_TOKENS / (n_val_files * 1150)))
    with ThreadPoolExecutor(n_val_files) as ex:
        texts = [t for ts in ex.map(lambda i: read_web(web_pfs[i], web_val_groups[web_idx[i]], C.WEB_MIN_SCORE)[:per_file],
                                    range(n_val_files)) for t in ts]
    write("web_val", texts)
    try:
        write("wiki_val", [wikitext_test()])
    except Exception as e:
        print(f"wikitext out-of-domain set skipped ({type(e).__name__}: {e})")
    print({k: f"{v:,}" for k, v in meta["tokens"].items()}, "validation tokens")

    # ---- training mix: Cosmopedia first, then quality-filtered FineWeb-Edu; windows are shuffled at training time ----
    train = open(P["train"] + ".tmp", "wb")
    cosmo_idx, cosmo_pfs = open_files("cosmopedia-v2", COSMO_FILES, C.COSMO_N_FILES)
    rounds = int(np.ceil(1.1 * C.COSMO_TOKENS / (len(cosmo_pfs) * 1000 * 700))) + 1
    groups = []
    for fi, pf in zip(cosmo_idx, cosmo_pfs):
        n = pf.metadata.num_row_groups
        groups.append(spread(n, rounds, {val_groups[fi]} if fi in val_groups else ()))
    print(f"Cosmopedia: {len(cosmo_pfs)} files x up to {min(len(g) for g in groups)} row-groups")
    preview = None
    got = 0
    for texts in rounds_iter(cosmo_pfs, groups, read_cosmo):
        preview = preview or texts[0][:300]
        got += write("train_cosmo", texts, count=True, f=train)
        print(f"\r  cosmopedia {got / 1e6:8.1f}M / {C.COSMO_TOKENS / 1e6:.0f}M tokens | {hms(time.time() - t0)}", end="", flush=True)
        if got >= C.COSMO_TOKENS:
            break
    print()

    web_idx2, web_pfs2 = web_idx, web_pfs
    keep_rate, avg = 0.41, 1150     # measured on FineWeb-Edu-dedup: ~41% of documents score >= 3.0
    rounds = int(np.ceil(1.1 * C.WEB_TOKENS / (len(web_pfs2) * 1000 * keep_rate * avg))) + 1
    groups = [spread(pf.metadata.num_row_groups, rounds, {web_val_groups[fi]}) for fi, pf in zip(web_idx2, web_pfs2)]
    print(f"FineWeb-Edu (score >= {C.WEB_MIN_SCORE}): {len(web_pfs2)} files x up to {min(len(g) for g in groups)} row-groups")
    got = 0
    reader = lambda pf, gi: read_web(pf, gi, C.WEB_MIN_SCORE)
    for texts in rounds_iter(web_pfs2, groups, reader):
        got += write("train_web", texts, count=True, f=train)
        print(f"\r  fineweb-edu {got / 1e6:8.1f}M / {C.WEB_TOKENS / 1e6:.0f}M tokens | {hms(time.time() - t0)}", end="", flush=True)
        if got >= C.WEB_TOKENS:
            break
    print()
    train.close()
    os.replace(P["train"] + ".tmp", P["train"])
    np.save(P["counts"], counts)
    meta["seconds"] = round(time.time() - t0)
    json.dump(meta, open(P["meta"], "w"))
    print(f"done in {hms(time.time() - t0)}: " + ", ".join(f"{k} {v:,}" for k, v in meta["tokens"].items()))
    print("--- first Cosmopedia document ---\n" + (preview or "") + "\n---")


# ------------------------------------------------------------------ instruction data (smol-smoltalk)
def first_exchange(messages):
    # (user, assistant) of a conversation's first turn; a leading system message is dropped
    ms = [m for m in messages if m["role"] != "system"]
    if len(ms) >= 2 and ms[0]["role"] == "user" and ms[1]["role"] == "assistant":
        return ms[0]["content"].strip(), ms[1]["content"].strip()
    return None


def prepare_sft(C, tok, eos, teacher_vocab):
    """Tokenize the first exchange of smol-smoltalk conversations into sft_raw.npz (UNK filtering waits for the vocab)."""
    P = paths(C.DATA_DIR)
    if os.path.exists(P["sft"]):
        print("sft_raw.npz already on disk")
        return
    import pyarrow.parquet as pq
    from huggingface_hub import HfFileSystem
    fs = HfFileSystem()
    ids, plen, tlen, seen = [], [], [], 0
    for fi in range(4):
        pf = retry(lambda: pq.ParquetFile(fs.open(f"datasets/HuggingFaceTB/smol-smoltalk/data/train-{fi:05d}-of-00004.parquet", "rb")))
        for gi in range(pf.metadata.num_row_groups):
            t = read_group(pf, gi, ["messages"])
            for msgs in (t.column("messages").to_pylist() if t is not None else []):
                ex = first_exchange(msgs)
                seen += 1
                if ex is None:
                    continue
                p = tok(PROMPT_TEMPLATE.format(query=ex[0]))["input_ids"]
                r = tok(ex[1])["input_ids"] + [eos]
                if len(r) < 2 or len(p) + len(r) > C.SFT_MAX_LEN + 1:
                    continue
                ids.append(np.asarray(p + r, dtype=np.uint32)); plen.append(len(p)); tlen.append(len(p) + len(r))
            if len(ids) >= C.SFT_MAX_EXAMPLES:
                break
        if len(ids) >= C.SFT_MAX_EXAMPLES:
            break
    n = min(len(ids), C.SFT_MAX_EXAMPLES)
    np.savez(P["sft"], ids=np.concatenate(ids[:n]), plen=np.array(plen[:n], np.int32), tlen=np.array(tlen[:n], np.int32))
    print(f"instruction data: kept {n:,} of {seen:,} conversations (first exchange, <= {C.SFT_MAX_LEN} tokens), "
          f"{sum(tlen[:n]) / 1e6:.1f}M tokens")


# ------------------------------------------------------------------ vocabulary
def build_vocab(C, tok, eos, teacher_vocab):
    P = paths(C.DATA_DIR)
    counts = np.load(P["counts"])
    sft = np.load(P["sft"])
    sft_counts = np.bincount(sft["ids"], minlength=teacher_vocab)
    weighted = counts + C.SFT_VOCAB_SHARE * counts.sum() / max(1, sft_counts.sum()) * sft_counts
    keep = np.argsort(-weighted, kind="stable")[: C.VOCAB_SIZE - 1]
    if eos not in keep:
        keep[-1] = eos
    keep = np.sort(keep)
    cov = lambda c: c[keep].sum() / max(1, c.sum())
    print(f"distinct teacher tokens seen: {(weighted > 0).sum():,} | keeping {len(keep):,} + UNK")
    print(f"token coverage  train {cov(counts):.3%} | instruction data {cov(sft_counts):.3%}")
    for name in ("cosmo_val", "web_val", "wiki_val"):
        if os.path.exists(P[name]):
            c = np.bincount(np.fromfile(P[name], dtype=np.uint32), minlength=teacher_vocab)
            print(f"token coverage  {name:9s} {cov(c):.3%}  (UNK rate {1 - cov(c):.2%})")
    json.dump({"teacher": C.TEACHER_NAME, "teacher_vocab_size": int(teacher_vocab), "eos_teacher_id": int(eos),
               "teacher_ids": keep.tolist()}, open(C.VOCAB_PATH, "w"))


# ------------------------------------------------------------------ benchmarks
def prepare_bench(C):
    P = paths(C.DATA_DIR)
    if os.path.exists(P["bench"]):
        print("bench.json already on disk")
        return
    import pyarrow.parquet as pq
    from huggingface_hub import HfFileSystem
    fs = HfFileSystem()
    rng = np.random.RandomState(0)
    load = lambda p: retry(lambda: pq.read_table(fs.open("datasets/" + p, "rb"))).to_pylist()
    pick = lambda rows: [rows[i] for i in rng.permutation(len(rows))[: C.BENCH_N]]
    bench = {}

    def attempt(name, fn):
        try:
            bench[name] = fn()
            print(f"  {name}: {len(bench[name]['items'])} items")
        except Exception as e:
            print(f"  {name} skipped ({type(e).__name__}: {e})")

    def arc():
        items = [dict(ctx=f"Question: {r['question']}\nAnswer:", choices=[" " + t for t in r["choices"]["text"]],
                      label=r["choices"]["label"].index(r["answerKey"]))
                 for r in pick(load("allenai/ai2_arc/ARC-Easy/test-00000-of-00001.parquet"))]
        return dict(type="mc", items=items)

    def hellaswag():
        def pre(t):
            t = t.strip().replace(" [title]", ". ")
            return re.sub(r"\[.*?\]", "", t).replace("  ", " ")
        items = [dict(ctx=pre(r["activity_label"] + ": " + r["ctx_a"] + " " + r["ctx_b"].capitalize()),
                      choices=[" " + pre(e) for e in r["endings"]], label=int(r["label"]))
                 for r in pick(load("Rowan/hellaswag/data/validation-00000-of-00001.parquet"))]
        return dict(type="mc", items=items)

    def sciq():
        items = []
        for r in pick(load("allenai/sciq/data/test-00000-of-00001.parquet")):
            if len(r["support"]) > 1200:
                continue
            ch = [r["distractor1"], r["distractor2"], r["distractor3"], r["correct_answer"]]
            items.append(dict(ctx=f"{r['support'].strip()}\nQuestion: {r['question']}\nAnswer:".strip(),
                              choices=[" " + c for c in ch], label=3))
        return dict(type="mc", items=items)

    def lambada():
        f = retry(lambda: fs.open("datasets/EleutherAI/lambada_openai/data/lambada_test.jsonl", "rb"))
        rows = [json.loads(line)["text"] for line in f.read().decode().splitlines() if line.strip()]
        items = [dict(ctx=t.rsplit(" ", 1)[0], target=" " + t.rsplit(" ", 1)[1]) for t in [rows[i] for i in rng.permutation(len(rows))[: C.BENCH_N]]]
        return dict(type="lm", items=items)

    for name, fn in (("arc_easy", arc), ("hellaswag", hellaswag), ("sciq", sciq), ("lambada", lambada)):
        attempt(name, fn)
    json.dump(bench, open(P["bench"], "w"))

In [ ]:
%%writefile hybrid50m_code/train.py
"""Training entry point: one process, or one process per GPU under torchrun (DDP).

    torchrun --nproc_per_node=2 train.py --cfg cfg.json --stage pretrain     # main run (resumes from its checkpoint)
    torchrun --nproc_per_node=2 train.py --cfg cfg.json --stage pilot        # equal-wall-clock arms (KD vs CE, Muon vs AdamW)
    torchrun --nproc_per_node=2 train.py --cfg cfg.json --stage sft          # instruction tuning

Every rank holds its own copy of the student and the teacher and reads its own slice of each global batch; gradients are
averaged by DDP. Rank 0 evaluates, checkpoints and logs. The data position is a pure function of the optimizer step.
"""
import argparse
import json
import math
import os
import signal
import sys
import time
from contextlib import nullcontext
from datetime import timedelta
from types import SimpleNamespace

import numpy as np
import torch
import torch.distributed as dist

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from core import (History, Teacher, TokenFile, VocabMap, WindowSampler, evaluate, export_final, hms, kd_loss, load_teacher_checked,
                  plot_history, prep_bench, read_json, run_bench, sample_text, save_checkpoint, split_batch, teacher_token_bytes,
                  PROMPT_TEMPLATE)
from data import paths
from model import Config, StudentLM
from optim import build_optimizer, lr_at

STOP = {"flag": False}


def _on_signal(*_):
    STOP["flag"] = True


class Env:
    def __init__(self, C):
        self.world = int(os.environ.get("WORLD_SIZE", 1))
        self.rank = int(os.environ.get("RANK", 0))
        self.local = int(os.environ.get("LOCAL_RANK", 0))
        if self.world > 1:
            dist.init_process_group("nccl" if torch.cuda.is_available() else "gloo", timeout=timedelta(minutes=120))
        if torch.cuda.is_available():
            torch.cuda.set_device(self.local)
            self.device = torch.device("cuda", self.local)
            torch.backends.cuda.matmul.allow_tf32 = True
        elif torch.backends.mps.is_available() and self.world == 1:
            self.device = torch.device("mps")
        else:
            self.device = torch.device("cpu")
        self.main = self.rank == 0
        prec = C.PRECISION
        if self.device.type == "cuda" and prec == "bf16" and torch.cuda.get_device_capability()[0] < 8:
            prec = "fp16"
        self.amp_dtype = {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": None}[prec] if self.device.type == "cuda" else None
        self.teacher_dtype = {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": torch.float32}[prec]
        if self.device.type == "cpu":
            self.teacher_dtype = torch.float32
        elif self.device.type == "mps":
            self.teacher_dtype = torch.float16

    def log(self, *a):
        if self.main:
            print(*a, flush=True)

    def amp(self):
        return torch.autocast("cuda", dtype=self.amp_dtype) if self.amp_dtype else nullcontext()

    def scaler(self):
        return torch.amp.GradScaler("cuda", enabled=self.amp_dtype == torch.float16)

    def barrier(self):
        if self.world > 1:
            dist.barrier()

    def any(self, flag):
        # True on every rank if it is True on any rank (stop decisions must be collective)
        if self.world == 1:
            return bool(flag)
        t = torch.tensor([1.0 if flag else 0.0], device=self.device)
        dist.all_reduce(t, op=dist.ReduceOp.MAX)
        return bool(t.item() > 0)

    def gpu_mem(self):
        return f"{torch.cuda.max_memory_allocated() / 1e9:.1f}GB" if self.device.type == "cuda" else "n/a"


class Context:
    """Everything the stages share: config, device, vocabulary, tokenizer, held-out sets, benchmarks, (lazy) teachers."""

    def __init__(self, C, env):
        from transformers import AutoTokenizer
        self.C, self.env = C, env
        self.P = paths(C.DATA_DIR)
        self.tok = AutoTokenizer.from_pretrained(C.TEACHER_NAME)
        self.vocab_json = json.load(open(C.VOCAB_PATH))
        self.vocab = VocabMap(self.vocab_json)
        self.vocab.set_bytes(teacher_token_bytes(self.tok, self.vocab_json["teacher_vocab_size"]))
        files = {n: TokenFile(self.P[f"{n}_val"], C.SEQ_LEN) for n in ("cosmo", "web", "wiki") if os.path.exists(self.P[f"{n}_val"])}
        self.sets = {n: f.fixed(C.VAL_WINDOWS) for n, f in files.items() if f.n_windows > 0}
        raw = read_json(self.P["bench"])
        self.bench = prep_bench(raw, self.tok) if raw and C.BENCH_EVERY > 0 else {}
        self._teachers = {}

    def teacher(self, name):
        if name not in self._teachers:
            self._teachers.clear()
            if self.env.device.type == "cuda":
                torch.cuda.empty_cache()
            probe = [self.sets["cosmo"][:2]]
            self._teachers[name] = load_teacher_checked(name, self.vocab, self.env.device, self.env.teacher_dtype, probe, self.env.log)
        return self._teachers[name]

    def new_model(self):
        return StudentLM(Config(vocab_size=self.vocab.size, **self.C.STUDENT))


# ------------------------------------------------------------------ the training loop (shared by all stages)
def run_job(ctx, J):
    """J: name, model, opt, scaler, teacher, alpha, get_batch(m), accum, tokens_per_step, total_steps, lr_fn, start_step, best,
    elapsed0, eval_fn(step, final), eval_every, save_every, milestone_every, minutes, hist, extra (saved in checkpoints)."""
    C, env = ctx.C, ctx.env
    model, opt, scaler = J.model, J.opt, J.scaler
    ddp = model
    if env.world > 1:
        ddp = torch.nn.parallel.DistributedDataParallel(model, device_ids=[env.local] if env.device.type == "cuda" else None)
    latest, best_path = (os.path.join(C.CKPT_DIR, f"{J.name}_{k}.pt") for k in ("latest", "best"))
    step, best, win, skipped, bad = J.start_step, J.best, [], 0, 0
    t0 = t_win = time.time()
    last_logged = step
    elapsed = J.elapsed = lambda: J.elapsed0 + time.time() - t0
    reason = None
    if env.device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    model.train()
    env.log(f"[{J.name}] steps {step}->{J.total_steps} | {J.tokens_per_step:,} tokens/step ({J.accum} micro-steps x {env.world} GPU) "
            f"| {'KD alpha ' + str(J.alpha) if J.teacher is not None else 'CE only (no teacher)'} | lr {J.lr_fn(step):.2e}")

    def checkpoint(path):
        if env.main:
            save_checkpoint(path, model, opt, step, ctx.vocab_json, scaler, best_val=best, elapsed=elapsed(), **J.extra)

    while step < J.total_steps:
        lr = J.lr_fn(step)
        for g in opt.param_groups:
            g["lr"] = lr
        sums = torch.zeros(3, device=env.device)
        for a in range(J.accum):
            xt, xs, y = J.get_batch(step * J.accum + a)
            with (ddp.no_sync() if env.world > 1 and a < J.accum - 1 else nullcontext()):
                t_logits = J.teacher.logits(xt) if J.teacher is not None else None
                with env.amp():
                    s_logits = ddp(xs)
                loss, kl, ce = kd_loss(s_logits, t_logits, y, C.TEMPERATURE, J.alpha if J.teacher is not None else 0.0)
                scaler.scale(loss / J.accum).backward()
            sums += torch.stack([loss.detach(), kl.detach(), ce.detach()]) / J.accum
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), C.GRAD_CLIP).item()
        scale_before = scaler.get_scale()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        step += 1
        skipped += int(scaler.get_scale() < scale_before)
        if env.world > 1:
            dist.all_reduce(sums)
            sums /= env.world
        loss_v, kl_v, ce_v = sums.tolist()
        bad = 0 if math.isfinite(loss_v) else bad + 1
        if bad >= 5:
            raise FloatingPointError("loss was NaN/inf for 5 steps in a row. The last checkpoint is intact; try PRECISION='fp32' "
                                     "(or 'bf16' on an Ampere+ GPU) or a lower LR")
        if math.isfinite(loss_v):
            win.append((loss_v, kl_v, ce_v, gnorm))

        if step in (1, 5) or step % C.LOG_EVERY == 0 or step == J.total_steps:
            now = time.time()
            w = np.nanmean(np.array(win), axis=0) if win else np.full(4, np.nan)
            tok_s = (step - last_logged) * J.tokens_per_step / max(1e-9, now - t_win)
            rem = (J.total_steps - step) * J.tokens_per_step / max(1.0, tok_s)
            if env.main:
                J.hist.add("train", dict(step=step, tokens=step * J.tokens_per_step, t=elapsed(), loss=w[0], kl=w[1], ce=w[2],
                                         gnorm=w[3], lr=lr, tok_s=tok_s, scale=scaler.get_scale()))
                print(f"step {step:6d}/{J.total_steps} ({100 * step / J.total_steps:4.1f}%) | {step * J.tokens_per_step / 1e6:7.1f}M tok "
                      f"| loss {w[0]:.3f} kl {w[1]:.3f} ce {w[2]:.3f} | lr {lr:.2e} | gnorm {w[3]:.2f} | {tok_s:,.0f} tok/s "
                      f"| elapsed {hms(elapsed())} | eta {hms(rem) if J.total_steps < 10 ** 8 else '-'} | mem {env.gpu_mem()}"
                      + (f" | loss-scale {scaler.get_scale():.0f} skipped {skipped}" if env.amp_dtype == torch.float16 else ""), flush=True)
            win, last_logged, t_win = [], step, now

        # collective stop decision: session time limit, a signal, or this arm's wall-clock budget
        over_session = (time.time() - C.NB_START) / 3600 > C.MAX_RUN_HOURS
        over_budget = J.minutes is not None and elapsed() / 60 >= J.minutes
        stop = env.any(over_session or over_budget or STOP["flag"])
        if stop and step < J.total_steps:
            reason = "budget" if over_budget and not (over_session or STOP["flag"]) else ("signal" if STOP["flag"] else "time")
        final = step == J.total_steps or reason is not None

        if step % J.eval_every == 0 or final:
            if env.main:
                metrics, lines, score = J.eval_fn(step, final)
                J.hist.add("eval", dict(step=step, tokens=step * J.tokens_per_step, t=elapsed(), **metrics))
                print("=" * 100 + f"\nEVAL @ step {step}/{J.total_steps}  " + " | ".join(f"{k} {v:.3f}" for k, v in metrics.items() if k.endswith(("/ce", "/bpb")))
                      + "".join("\n  " + ln for ln in lines) + "\n" + "=" * 100, flush=True)
                if score < best:
                    best = score
                    save_checkpoint(best_path, model, None, step, ctx.vocab_json, best_val=best, **J.extra)
                J.hist.save()
                if plot_history(J.hist.h, os.path.join(C.PERSIST_DIR, f"dashboard_{J.name}.png"), J.name):
                    print("DASHBOARD " + os.path.join(C.PERSIST_DIR, f"dashboard_{J.name}.png"), flush=True)
            env.barrier()
            t_win = time.time()
        if step % J.save_every == 0 or final:
            checkpoint(latest)
            if env.main:
                J.hist.save()
        if J.milestone_every and step % J.milestone_every == 0 and env.main:
            save_checkpoint(os.path.join(C.CKPT_DIR, f"{J.name}_step{step}.pt"), model, None, step, ctx.vocab_json, **J.extra)
        if reason is not None:
            break

    status = "paused" if reason in ("time", "signal") else "done"
    if env.main:
        J.hist.save()
        write_state(ctx, J.name, status, step, J.total_steps, best, elapsed(), J.tokens_per_step, reason)
        print(f"[{J.name}] {status.upper()} at step {step}/{J.total_steps}" + (f" ({reason})" if reason else "") + f" | best val bpb {best:.4f}", flush=True)
    env.barrier()
    return status


def write_state(ctx, name, status, step, total, best, elapsed, tokens_per_step, reason=None):
    if ctx.env.main:
        json.dump(dict(name=name, status=status, reason=reason, step=step, total_steps=total, best=best, elapsed=elapsed,
                       tokens=step * tokens_per_step, world=ctx.env.world, tokens_per_step=tokens_per_step),
                  open(os.path.join(ctx.C.CKPT_DIR, f"state_{name}.json"), "w"))


def resume_or_new(ctx, name, muon, lr, from_ckpt=None):
    """Model + optimizer + scaler, restored from `<name>_latest.pt` if it exists (else optionally initialised from `from_ckpt`)."""
    C, env = ctx.C, ctx.env
    latest = os.path.join(C.CKPT_DIR, f"{name}_latest.pt")
    model, ck = ctx.new_model(), None
    start, best, elapsed0 = 0, float("inf"), 0.0
    src = latest if os.path.exists(latest) else from_ckpt
    if src:
        ck = torch.load(src, map_location="cpu", weights_only=False)
        assert ck["vocab"]["teacher_ids"] == ctx.vocab_json["teacher_ids"], f"{src} was trained with a different vocab"
        model = StudentLM(Config.from_dict(ck["config"]))
        model.load_state_dict(ck["model"])
    model.to(env.device)
    opt, scaler = build_optimizer(model, lr, C.WEIGHT_DECAY, muon), env.scaler()
    if src == latest:
        if ck.get("muon", muon) != muon:
            raise RuntimeError(f"{latest} was trained with USE_MUON={ck.get('muon')}; the config says {muon}. Change the config back.")
        opt.load_state_dict(ck["optimizer"])
        if ck.get("scaler"):
            scaler.load_state_dict(ck["scaler"])
        start, best, elapsed0 = ck["step"], ck.get("best_val", best), ck.get("elapsed", 0.0)
        env.log(f"[{name}] RESUMING from step {start} ({ck.get('elapsed', 0) / 3600:.1f} h of training so far; best val bpb {best:.4f})")
    else:
        env.log(f"[{name}] starting from " + (f"{src} (step {ck['step']})" if src else "scratch") + f" | {model.num_params() / 1e6:.2f}M params")
    return model, opt, scaler, start, best, elapsed0, ck


# ------------------------------------------------------------------ stage 1: pretraining (also the pilot arms)
def run_pretrain(ctx, name, kd, muon, minutes=None, total_steps=None, sched=None, eval_every=None):
    C, env = ctx.C, ctx.env
    total = total_steps or C.PRETRAIN_MAX_STEPS
    accum = C.TOKENS_PER_STEP // (C.BATCH_SIZE * C.SEQ_LEN * env.world)
    assert accum * C.BATCH_SIZE * C.SEQ_LEN * env.world == C.TOKENS_PER_STEP, \
        "TOKENS_PER_STEP must be a multiple of BATCH_SIZE * SEQ_LEN * number of GPUs"
    latest = os.path.join(C.CKPT_DIR, f"{name}_latest.pt")
    if C.REQUIRE_CHECKPOINT and not os.path.exists(latest):
        raise FileNotFoundError(f"{latest} not found and REQUIRE_CHECKPOINT=True: attach the previous run's output as an input")
    model, opt, scaler, start, best, elapsed0, _ = resume_or_new(ctx, name, muon, C.LR)
    if start >= total:
        env.log(f"[{name}] already finished (step {start}/{total})")
        write_state(ctx, name, "done", start, total, best, elapsed0, C.TOKENS_PER_STEP)
        return "done"
    teacher = ctx.teacher(C.TEACHER_NAME) if kd else None
    train = TokenFile(ctx.P["train"], C.SEQ_LEN)
    sampler = WindowSampler(train.n_windows, C.SEED)
    kind = sched or C.SCHEDULE
    hist = History(os.path.join(C.CKPT_DIR, f"history_{name}.json"), start)
    env.log(f"[{name}] {train.n_windows:,} training windows ({train.n_tokens / 1e6:.0f}M tokens); {total * C.TOKENS_PER_STEP / 1e6:,.0f}M tokens "
            f"planned = {total * C.TOKENS_PER_STEP / train.n_tokens:.2f} epochs | schedule {kind} | optimizer {'Muon+AdamW' if muon else 'AdamW'}")
    if env.main:
        json.dump(dict(params=model.num_params(), config=model.cfg.to_dict(), world=env.world, device=str(env.device),
                       gpu=torch.cuda.get_device_name() if env.device.type == "cuda" else env.device.type,
                       precision=C.PRECISION, kd=kd, muon=muon, tokens_per_step=C.TOKENS_PER_STEP, total_steps=total),
                  open(os.path.join(C.CKPT_DIR, f"info_{name}.json"), "w"))

    def get_batch(m):
        idx = sampler.take((m * env.world + env.rank) * C.BATCH_SIZE, C.BATCH_SIZE)
        return split_batch(train.get(idx), ctx.vocab, env.device)

    def eval_fn(step, final):
        m, lines = {}, []
        for nm, w in ctx.sets.items():
            r = evaluate(model, ctx.vocab, w, env.device, env.amp, teacher, C.EVAL_BS)
            m.update({f"{nm}/{k}": v for k, v in r.items()})
            lines.append(f"{nm:6s} CE {r['ce']:.3f} ppl {r['ppl']:.1f} bpb {r['bpb']:.4f}"
                         + (f" | teacher CE {r['t_ce']:.3f} bpb {r['t_bpb']:.4f} | KL {r['kl']:.3f} | top-1 agree {r['agree']:.1f}%" if teacher is not None else ""))
        if ctx.bench and (final or step % C.BENCH_EVERY == 0):
            b = run_bench(model, ctx.vocab, ctx.bench, env.device, env.amp, C.SEQ_LEN - 1)
            J.hist.add("bench", dict(step=step, tokens=step * C.TOKENS_PER_STEP, t=J.elapsed(), **b))
            lines.append("bench  " + " | ".join(f"{k} {v:.1f}" for k, v in b.items()))
        for p in C.SAMPLE_PROMPTS:
            lines.append(f"> {p!r} -> {sample_text(model, ctx.vocab, ctx.tok, p, env.device, env.amp, 48, 0.7, 40, 1.15)!r}")
        score = float(np.mean([m[f"{nm}/bpb"] for nm in ("cosmo", "web") if f"{nm}/bpb" in m]))
        return m, lines, score

    J = SimpleNamespace(name=name, model=model, opt=opt, scaler=scaler, teacher=teacher, alpha=C.ALPHA, get_batch=get_batch, accum=accum,
                        tokens_per_step=C.TOKENS_PER_STEP, total_steps=total, lr_fn=lambda s: lr_at(s, total, C.LR, C.WARMUP, kind, C.DECAY_FRAC),
                        start_step=start, best=best, elapsed0=elapsed0, eval_fn=eval_fn, eval_every=eval_every or C.EVAL_EVERY,
                        save_every=C.SAVE_EVERY, milestone_every=C.MILESTONE_EVERY if name == "pretrain" else 0, minutes=minutes,
                        hist=hist, extra=dict(muon=muon, kd=kd))
    status = run_job(ctx, J)
    del model, opt
    return status


def run_pilot(ctx):
    C, env = ctx.C, ctx.env
    for arm in C.PILOT_ARMS:
        name = f"pilot_{arm['name']}"
        if os.path.exists(os.path.join(C.CKPT_DIR, f"state_{name}.json")) and read_json(os.path.join(C.CKPT_DIR, f"state_{name}.json"))["status"] == "done":
            env.log(f"[{name}] already done")
            continue
        status = run_pretrain(ctx, name, arm["kd"], arm["muon"], minutes=C.PILOT_MINUTES, total_steps=10 ** 9, sched="constant",
                              eval_every=C.PILOT_EVAL_EVERY)
        if env.device.type == "cuda":
            torch.cuda.empty_cache()
        if status == "paused":
            break


# ------------------------------------------------------------------ stage 2: instruction tuning
def load_sft(ctx):
    C, env = ctx.C, ctx.env
    z = np.load(ctx.P["sft"])
    ids, plen, tlen = z["ids"], z["plen"].astype(np.int64), z["tlen"].astype(np.int64)
    starts = np.cumsum(tlen) - tlen
    unk = (ctx.vocab.t2s[torch.from_numpy(ids.astype(np.int64))] == ctx.vocab.unk_id).numpy().astype(np.int64)
    cs = np.concatenate([[0], np.cumsum(unk)])
    frac = (cs[starts + tlen] - cs[starts + plen]) / np.maximum(1, tlen - plen)
    keep = np.where(frac <= C.SFT_MAX_UNK_FRAC)[0]
    env.log(f"[sft] {len(keep):,} of {len(plen):,} examples kept (<= {C.SFT_MAX_UNK_FRAC:.0%} unknown tokens in the answer)")
    rng = np.random.RandomState(C.SEED)
    keep = keep[rng.permutation(len(keep))]
    n_val = max(1, min(2000, len(keep) // 50))
    return ids, starts, plen, tlen, keep[:n_val], keep[n_val:]


def run_sft(ctx):
    C, env = ctx.C, ctx.env
    ids, starts, plen, tlen, val_idx, train_idx = load_sft(ctx)
    eos, B = ctx.vocab_json["eos_teacher_id"], C.SFT_BATCH_SIZE

    def collate(idx):
        T = int(max(tlen[i] for i in idx)) - 1
        x = torch.full((len(idx), T), eos, dtype=torch.long)
        y = torch.full((len(idx), T), -100, dtype=torch.long)
        for k, i in enumerate(idx):
            full = torch.from_numpy(ids[starts[i]: starts[i] + tlen[i]].astype(np.int64))
            x[k, : len(full) - 1] = full[:-1]
            y[k, plen[i] - 1: len(full) - 1] = full[plen[i]:]
        x, y = x.to(env.device), y.to(env.device)
        lab = ctx.vocab.to_student(y.clamp(min=0))
        lab[(y == -100) | (lab == ctx.vocab.unk_id)] = -100
        return x, ctx.vocab.to_student(x), lab

    order = train_idx[np.argsort(tlen[train_idx], kind="stable")]       # length-bucketed batches: little padding
    chunks = [order[i: i + B] for i in range(0, len(order) - B + 1, B)]
    n_chunks = len(chunks)
    accum = C.SFT_GRAD_ACCUM
    total = n_chunks * C.SFT_EPOCHS // (accum * env.world)
    avg_len = float(np.mean(tlen[train_idx]))
    tokens_per_step = int(B * accum * env.world * avg_len)
    perms = {}

    def get_batch(m):
        p = m * env.world + env.rank
        ep, i = divmod(p, n_chunks)
        if ep not in perms:
            perms.clear()
            perms[ep] = np.random.RandomState(C.SEED + ep).permutation(n_chunks)
        return collate(chunks[perms[ep][i]])

    pre_best = os.path.join(C.CKPT_DIR, "pretrain_best.pt")
    pre = torch.load(pre_best, map_location="cpu", weights_only=False) if os.path.exists(pre_best) else {}
    muon = pre.get("muon", C.USE_MUON)
    model, opt, scaler, start, best, elapsed0, _ = resume_or_new(ctx, "sft", muon, C.SFT_LR, from_ckpt=pre_best)
    if start >= total:
        env.log(f"[sft] already finished (step {start}/{total})")
        write_state(ctx, "sft", "done", start, total, best, elapsed0, tokens_per_step)
        return "done"
    teacher = ctx.teacher(C.SFT_TEACHER) if C.SFT_ALPHA > 0 else None
    env.log(f"[sft] {len(train_idx):,} train / {len(val_idx)} val examples, {n_chunks:,} batches of {B}; {total:,} steps over {C.SFT_EPOCHS} epochs "
            f"(~{total * tokens_per_step / 1e6:.0f}M tokens)")
    hist = History(os.path.join(C.CKPT_DIR, "history_sft.json"), start)

    def eval_fn(step, final):
        model.eval()
        ce_s = kl_s = 0.0
        nb = 0
        with torch.no_grad():
            for i in range(0, len(val_idx), B):
                xt, xs, y = collate(val_idx[i: i + B])
                with env.amp():
                    s = model(xs)
                _, kl, ce = kd_loss(s, teacher.logits(xt) if teacher is not None else None, y, 1.0, 1.0 if teacher is not None else 0.0)
                ce_s += ce.item(); kl_s += kl.item(); nb += 1
        model.train()
        m = {"sft/ce": ce_s / nb, "sft/kl": kl_s / nb}
        lines = [f"sft val CE {m['sft/ce']:.3f} ppl {math.exp(m['sft/ce']):.1f}" + (f" | KL {m['sft/kl']:.3f}" if teacher is not None else "")]
        for q in C.SAMPLE_QUERIES:
            a = sample_text(model, ctx.vocab, ctx.tok, PROMPT_TEMPLATE.format(query=q), env.device, env.amp, 80, 0.6, 40, 1.15).strip()
            lines.append(f"Q: {q}\n    A: {a!r}")
        if ctx.bench and final:
            b = run_bench(model, ctx.vocab, ctx.bench, env.device, env.amp, C.SEQ_LEN - 1)
            hist.add("bench", dict(step=step, tokens=step * tokens_per_step, t=J.elapsed(), **b))
            lines.append("bench  " + " | ".join(f"{k} {v:.1f}" for k, v in b.items()))
        return m, lines, m["sft/ce"]

    J = SimpleNamespace(name="sft", model=model, opt=opt, scaler=scaler, teacher=teacher, alpha=C.SFT_ALPHA, get_batch=get_batch, accum=accum,
                        tokens_per_step=tokens_per_step, total_steps=total, lr_fn=lambda s: lr_at(s, total, C.SFT_LR, C.SFT_WARMUP, "cosine"),
                        start_step=start, best=best, elapsed0=elapsed0, eval_fn=eval_fn, eval_every=C.SFT_EVAL_EVERY, save_every=C.SFT_SAVE_EVERY,
                        milestone_every=0, minutes=None, hist=hist, extra=dict(muon=muon, kd=teacher is not None))
    return run_job(ctx, J)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cfg", required=True)
    ap.add_argument("--stage", required=True, choices=["pretrain", "pilot", "sft"])
    args = ap.parse_args()
    C = SimpleNamespace(**json.load(open(args.cfg)))
    signal.signal(signal.SIGINT, _on_signal)
    signal.signal(signal.SIGTERM, _on_signal)
    env = Env(C)
    torch.manual_seed(C.SEED); np.random.seed(C.SEED)
    env.log(f"device {env.device} | {env.world} process(es) | student autocast {env.amp_dtype} | teacher {env.teacher_dtype}")
    ctx = Context(C, env)
    if args.stage == "pretrain":
        run_pretrain(ctx, "pretrain", C.KD, C.USE_MUON)
    elif args.stage == "pilot":
        run_pilot(ctx)
    else:
        run_sft(ctx)
    if env.world > 1:
        dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile hybrid50m_code/selftest.py
"""Cheap correctness checks for the pieces the whole run depends on. `python hybrid50m/selftest.py` (the notebook runs it too)."""
import math
import os
import sys

import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from core import WindowSampler, kd_loss, teacher_token_bytes
from model import Config, StudentLM, from_v4_state_dict
from optim import MuonAdamW, build_optimizer, lr_at, newton_schulz


def ok(msg):
    print("  ok:", msg)


def test_model(old_model_cls=None, old_cfg_cls=None):
    torch.manual_seed(0)
    cfg = Config()
    m = StudentLM(cfg)
    n = m.num_params()
    assert 49e6 < n < 54e6, n
    ok(f"default config has {n / 1e6:.2f}M parameters (GQA {cfg.n_heads}:{cfg.n_kv_heads}, {cfg.n_layers} layers, QK-norm)")

    small = Config(vocab_size=500, d_model=64, n_layers=3, n_heads=4, n_kv_heads=2, ffn_hidden=96, max_seq_len=32, emb_dim=16)
    m = StudentLM(small).eval()
    x = torch.randint(0, 500, (2, 16))
    y1 = m(x)
    x2 = x.clone(); x2[:, 10:] = torch.randint(0, 500, (2, 6))
    assert torch.allclose(y1[:, :10], m(x2)[:, :10], atol=1e-5)
    assert not torch.allclose(y1[:, 10:], m(x2)[:, 10:], atol=1e-3)
    ok("causal: changing future tokens leaves earlier logits untouched")

    shared = StudentLM(Config(**{**small.to_dict(), "block_repeats": 2}))
    assert shared.num_params() == StudentLM(small).num_params()
    ok("block_repeats=2 doubles depth with identical parameter count")

    if old_model_cls is not None:
        ocfg = old_cfg_cls(vocab_size=500, d_model=64, n_layers=3, n_heads=4, ffn_hidden=96, max_seq_len=32, emb_dim=16)
        old = old_model_cls(ocfg).eval()
        new = StudentLM(Config.from_v4(ocfg.to_dict())).eval()
        new.load_state_dict(from_v4_state_dict(old.state_dict(), new.cfg), strict=False)
        assert torch.allclose(old(x), new(x), atol=1e-5), (old(x) - new(x)).abs().max()
        ok("loads the 10M model's weights (fused qkv -> q/k/v) and reproduces its logits exactly")


def test_optim():
    torch.manual_seed(0)
    cfg = Config(vocab_size=200, d_model=32, n_layers=2, n_heads=4, n_kv_heads=2, ffn_hidden=48, max_seq_len=16, emb_dim=8)
    a, b = StudentLM(cfg), StudentLM(cfg)
    b.load_state_dict(a.state_dict())
    oa = build_optimizer(a, 1e-2, 0.1, muon=False)
    decay = [p for p in b.parameters() if p.dim() >= 2]
    plain = [p for p in b.parameters() if p.dim() < 2]
    ob = torch.optim.AdamW([{"params": decay, "weight_decay": 0.1}, {"params": plain, "weight_decay": 0.0}], lr=1e-2, betas=(0.9, 0.95))
    for _ in range(5):
        x = torch.randint(0, 200, (4, 16))
        for m, o in ((a, oa), (b, ob)):
            o.zero_grad(); m(x).logsumexp(-1).mean().backward(); o.step()
    # rounding differences accumulate to ~3e-6 (parameters move ~5e-2 in total); a real bug (e.g. no bias correction) is off by ~1e-2
    worst = max((p - q).abs().max().item() for p, q in zip(a.parameters(), b.parameters()))
    assert worst < 5e-5, worst
    ok(f"AdamW half matches torch.optim.AdamW over 5 steps (max difference {worst:.1e})")

    g = torch.randn(40, 100)
    s = torch.linalg.svdvals(newton_schulz(g))
    assert s.min() > 0.4 and s.max() < 1.6, (s.min(), s.max())
    ok(f"Newton-Schulz pushes singular values to ~1 (range {s.min():.2f}..{s.max():.2f})")

    c = StudentLM(cfg)
    oc = build_optimizer(c, 1e-2, 0.1, muon=True)
    kinds = [g_["kind"] for g_ in oc.param_groups]
    assert kinds.count("muon") == 1 and all(p.dim() == 2 and n.startswith("blocks.") for n, p in c.named_parameters() if any(p is q for q in oc.param_groups[0]["params"]))
    before = [p.clone() for p in c.parameters()]
    oc.zero_grad(); c(torch.randint(0, 200, (4, 16))).logsumexp(-1).mean().backward(); oc.step()
    assert all(not torch.equal(p, q) for p, q in zip(before, c.parameters()))
    oc.load_state_dict(oc.state_dict())
    ok("Muon covers only block matrices, updates every parameter, and its state round-trips")


def test_schedule_and_sampler():
    assert lr_at(0, 1000, 1.0, 10) == 0.1 and lr_at(9, 1000, 1.0, 10) == 1.0
    assert lr_at(500, 1000, 1.0, 10) == 1.0 and lr_at(799, 1000, 1.0, 10) == 1.0
    assert lr_at(1000, 1000, 1.0, 10) == 0.02 and 0.02 < lr_at(900, 1000, 1.0, 10) < 1.0
    assert lr_at(100, 10 ** 9, 1.0, 10, "constant") == 1.0
    ok("warmup-stable-decay schedule: flat until 80%, then 1-sqrt decay to 2%")

    s = WindowSampler(37, seed=3)
    whole = s.take(0, 100)
    assert (np.concatenate([s.take(0, 30), s.take(30, 70)]) == whole).all()
    assert sorted(whole[:37].tolist()) == list(range(37)) and sorted(whole[37:74].tolist()) == list(range(37))
    assert not (whole[:37] == whole[37:74]).all()
    ok("sampler: each epoch visits every window once, any resume point continues the same stream")


def test_loss():
    torch.manual_seed(0)
    s = torch.randn(2, 5, 20)
    y = torch.randint(0, 20, (2, 5)); y[0, 0] = -100
    total, kl, ce = kd_loss(s, s.clone(), y, alpha=0.7)
    assert kl.abs() < 1e-5 and torch.allclose(total, 0.3 * ce, atol=1e-5)
    _, kl2, _ = kd_loss(s, torch.randn(2, 5, 20), y, alpha=0.7)
    assert kl2 > 0.1
    ok("KD loss: zero KL against itself, positive against a different teacher, ignored targets skipped")


def test_bytes(tokenizer):
    tb = teacher_token_bytes(tokenizer, max(len(tokenizer), 151936))
    for text in ["Hello, world! The cat sat on the mat.", "naïve café — 日本語のテキスト 😀", "def f(x):\n    return x ** 2\n"]:
        ids = tokenizer(text)["input_ids"]
        assert int(sum(tb[i] for i in ids)) == len(text.encode()), (text, sum(tb[i] for i in ids), len(text.encode()))
    assert tb[tokenizer.eos_token_id] == 0
    ok("byte lengths of tokens sum to the UTF-8 length of the text (so bits-per-byte is exact)")


def main(with_tokenizer=True):
    print("self-test")
    old_cls = old_cfg = None
    root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    if os.path.exists(os.path.join(root, "student_model.py")):
        sys.path.insert(0, root)
        from student_model import StudentConfig, StudentLM as Old
        old_cls, old_cfg = Old, StudentConfig
    test_model(old_cls, old_cfg)
    test_optim()
    test_schedule_and_sampler()
    test_loss()
    if with_tokenizer:
        try:
            from transformers import AutoTokenizer
            test_bytes(AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B"))
        except Exception as e:
            print(f"  skipped byte-accounting test ({type(e).__name__}: {e})")
    print("all self-tests passed")


if __name__ == "__main__":
    main()

In [ ]:
import math, random, subprocess, signal, warnings
from contextlib import nullcontext
from types import SimpleNamespace
import numpy as np
import torch
warnings.filterwarnings("ignore")
sys.path.insert(0, CODE_DIR)
import model as M, optim as O, core as K, data as D, selftest
from transformers import AutoTokenizer, AutoConfig
tokenizer = AutoTokenizer.from_pretrained(TEACHER_NAME)
EOS = tokenizer.eos_token_id
TEACHER_VOCAB = max(AutoConfig.from_pretrained(TEACHER_NAME).vocab_size, len(tokenizer))

N_GPU = torch.cuda.device_count()
if N_GPU:
    DEVICE = torch.device("cuda")
    print(f"{N_GPU} x {torch.cuda.get_device_name()} | {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB each")
    if N_GPU == 1 and RUN_MODE in ("pilot", "pretrain", "sft"):
        print("NOTE: one GPU. It works (twice the accumulation steps) but takes twice as long; Kaggle: Settings -> Accelerator -> GPU T4 x2")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Apple GPU (MPS): fine for the smoke test, far too slow for the real run")
else:
    DEVICE = torch.device("cpu")
    print("no GPU" + ("" if RUN_MODE == "data" else ": the teacher will be extremely slow"))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

selftest.main()
cfg = M.Config(**STUDENT)
_m = M.StudentLM(cfg)
rows, total = {}, 0
import re
for n, p in _m.named_parameters():
    key = re.sub(r"blocks\.\d+\.", "blocks.*.", n)
    rows.setdefault(key, [tuple(p.shape), 0, 0]); rows[key][1] += 1; rows[key][2] += p.numel(); total += p.numel()
print(f"{'tensor':30s} {'shape':>14s} {'copies':>7s} {'params':>12s}")
for k, (shape, copies, cnt) in rows.items():
    print(f"{k:30s} {str(shape):>14s} {copies:>7d} {cnt:>12,}")
print(f"{'TOTAL':30s} {'':>14s} {'':>7s} {total:>12,} = {total / 1e6:.2f}M (fp16 file ~{total * 2 / 1e6:.0f} MB; the output head is tied to the embedding)")
print(f"tokens per parameter at {PRETRAIN_MAX_STEPS * TOKENS_PER_STEP / 1e9:.2f}B tokens: {PRETRAIN_MAX_STEPS * TOKENS_PER_STEP / total:.0f} (compute-optimal is ~20)")
del _m

## 1. Data (RUN_MODE `"data"` or `"smoke"`; no GPU needed)

- **Validation** (never read by training): Cosmopedia documents chosen exactly as in the 10M run, held-out FineWeb-Edu documents, and the WikiText-103 test set (out-of-domain).
- **Training mix**: Cosmopedia v2 row-groups spread over `COSMO_N_FILES` files, then FineWeb-Edu-dedup row-groups spread over `WEB_N_FILES` files, **keeping only documents whose educational score is >= `WEB_MIN_SCORE`**. Both are tokenized with the teacher's tokenizer into one flat `uint32` file; windows are shuffled at training time, which mixes the sources.
- **Instruction data**: the first exchange of each smol-smoltalk conversation (system prompts dropped), in the `### Question / ### Answer` format.
- **Vocabulary**: the `VOCAB_SIZE - 1` most frequent teacher tokens over the training mix (plus a small instruction-data share); everything else becomes `UNK`. The coverage printout is what to read: the UNK rate on **held-out** text is how much of real text the student can never see or say.
- **Benchmarks**: ARC-Easy, HellaSwag, SciQ, LAMBADA, as plain text in `bench.json`.

The output of this stage (`data/`, `vocab.json`) is what later sessions attach as an input: in Kaggle, *Save Version -> Save & Run All*, then *New Dataset from the notebook output*.

In [ ]:
if ("data" in STAGES) and not PAUSED:
    C = SimpleNamespace(**cfg_dict())
    t0 = time.time()
    D.prepare_training_data(C, tokenizer, EOS, TEACHER_VOCAB)
    D.prepare_sft(C, tokenizer, EOS, TEACHER_VOCAB)
    if not os.path.exists(VOCAB_PATH):
        D.build_vocab(C, tokenizer, EOS, TEACHER_VOCAB)
    else:
        print("using existing", VOCAB_PATH)
    print("benchmarks:")
    D.prepare_bench(C)
    print(f"data stage finished in {K.hms(time.time() - t0)}")
    for f in sorted(os.listdir(DATA_DIR)):
        print(f"  {f:22s} {os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6:10,.1f} MB")

In [ ]:
# every later stage needs these files (from the data stage of this session or attached as an input)
if RUN_MODE != "data":
    P = D.paths(DATA_DIR)
    missing = [os.path.basename(p) for p in (P["train"], P["cosmo_val"], P["web_val"], P["sft"], P["bench"], VOCAB_PATH) if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(f"missing {missing}. Run RUN_MODE='data' first and attach its output as an input to this notebook "
                                f"(or use RUN_MODE='smoke', which builds everything itself).")
    _v = json.load(open(VOCAB_PATH))
    assert len(_v["teacher_ids"]) + 1 == VOCAB_SIZE, f"vocab.json has {len(_v['teacher_ids']) + 1} entries but VOCAB_SIZE={VOCAB_SIZE}"
    print("data ready:", json.load(open(P["meta"])).get("tokens"))

## 2. Launcher

Training runs as `torchrun --nproc_per_node=<GPUs> train.py`, one process per GPU; its output streams here. The dashboard figure redraws in place at every evaluation. Interrupting the cell (stop button) sends the training a SIGINT: it finishes the step, writes a checkpoint and exits, so re-running the cell resumes.

How to read the eval block (printed every `EVAL_EVERY` steps):
- **CE / ppl / bpb** per held-out set: `cosmo` (in-distribution), `web` (held-out FineWeb-Edu), `wiki` (**out-of-domain**). Bits-per-byte is the number to compare across models and tokenizers.
- **teacher CE / bpb, KL, top-1 agree** (distillation runs): the gap to the teacher, how far the student's distribution is from the teacher's, and how often both pick the same next token.
- **bench**: zero-shot accuracies (%): chance is 25% for ARC-Easy, HellaSwag and SciQ. A 50M model should sit a little above chance on ARC/SciQ; the *trend* over training is the signal, and 1,000 examples carry about +-1.5% noise.
- **loss-scale / skipped**: fp16 loss scaling. A handful of skipped steps early on is normal; a steady stream is not.

In [ ]:
from IPython.display import display, Image


def warm_cache(path, chunk=64 << 20):
    # The trainer reads random 2 KB windows through a memmap. On Kaggle's network-backed /kaggle/input that is slow until the pages are cached
    # (throughput climbed from ~2.5k to ~7k tokens/s over the first ~500 steps of every session), so read the file once, sequentially, first.
    import time as _time
    t0, n = _time.time(), 0
    with open(path, "rb", buffering=0) as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            n += len(b)
    print(f"read {n / 1e9:.1f} GB of {os.path.basename(path)} in {_time.time() - t0:.0f}s to warm the page cache")


def launch(stage):
    # runs train.py (torchrun when there are 2+ GPUs), streams its output, returns the final state of the stage
    if stage in ("pilot", "pretrain"):
        warm_cache(P["train"])
    json.dump(cfg_dict(), open(CFG_PATH, "w"))
    train_py = os.path.join(CODE_DIR, "train.py")
    cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={N_GPU}", train_py] if N_GPU > 1 else [sys.executable, train_py]
    cmd += ["--cfg", CFG_PATH, "--stage", stage]
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false", "PYTHONPATH": CODE_DIR}
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env, cwd=CODE_DIR)
    handle = None
    try:
        for line in proc.stdout:
            if line.startswith("DASHBOARD "):
                img = Image(filename=line.split(" ", 1)[1].strip())
                if handle is None:
                    handle = display(img, display_id=True)
                else:
                    handle.update(img)
            else:
                print(line, end="")
    except KeyboardInterrupt:
        print("\ninterrupt: asking the trainer to checkpoint and stop ...")
        proc.send_signal(signal.SIGINT)
        try:
            proc.wait(timeout=600)
        except subprocess.TimeoutExpired:
            proc.kill()
    rc = proc.wait()
    if rc not in (0, -signal.SIGINT):
        raise RuntimeError(f"train.py exited with code {rc}; the last checkpoint is intact. Scroll up for the error.")


def stage_state(name):
    return K.read_json(os.path.join(CKPT_DIR, f"state_{name}.json"), {})

## 3. Pilot: what should the long run use?

Each arm trains for `PILOT_MINUTES` of wall-clock time at a constant learning rate on the same data in the same order, so the curves answer the real question: **at equal GPU time, which setup is better?**

- `kd_muon` vs `ce_muon`: is distilling from the 0.5B teacher worth its cost, or does plain training on more tokens win? (Distillation makes each token more expensive; the table below prints the measured exchange rate. A short pilot favours distillation, because it wins early: look at whether the gap is *shrinking* by the end.)
- `kd_muon` vs `kd_adamw`: does Muon help here?

Set `KD` / `USE_MUON` in the config from the result, then run `"pretrain"`.

In [ ]:
if ("pilot" in STAGES) and not PAUSED:
    launch("pilot")

In [ ]:
if ("pilot" in STAGES) and not PAUSED:
    import matplotlib.pyplot as plt
    arms = [a["name"] for a in PILOT_ARMS if os.path.exists(os.path.join(CKPT_DIR, f"history_pilot_{a['name']}.json"))]
    H = {a: K.read_json(os.path.join(CKPT_DIR, f"history_pilot_{a}.json")) for a in arms}
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
    for a in arms:
        ev = H[a]["eval"]
        if not ev:
            continue
        bpb = [np.mean([r[k] for k in ("cosmo/bpb", "web/bpb") if k in r]) for r in ev]
        ax[0].plot([r["t"] / 60 for r in ev], bpb, marker="o", label=a)
        ax[1].plot([r["tokens"] / 1e6 for r in ev], bpb, marker="o", label=a)
        tr = H[a]["train"]
        ax[2].plot([r["t"] / 60 for r in tr], [r["tok_s"] / 1e3 for r in tr], label=a)
    ax[0].set_title("held-out bits/byte vs wall-clock (lower is better)"); ax[0].set_xlabel("minutes")
    ax[1].set_title("... vs tokens seen"); ax[1].set_xlabel("M tokens")
    ax[2].set_title("throughput (k tokens/s)"); ax[2].set_xlabel("minutes")
    for a_ in ax:
        a_.grid(alpha=0.3); a_.legend()
    plt.tight_layout(); plt.show()

    print(f"{'arm':10s} {'minutes':>8s} {'M tokens':>9s} {'tok/s':>8s} {'cosmo bpb':>10s} {'web bpb':>8s} {'wiki bpb':>9s}")
    final, speed = {}, {}
    for a in arms:
        ev, tr = H[a]["eval"], H[a]["train"]
        if not ev:
            continue
        r = ev[-1]
        final[a] = np.mean([r[k] for k in ("cosmo/bpb", "web/bpb") if k in r])
        speed[a] = np.mean([x['tok_s'] for x in tr[-10:]])
        print(f"{a:10s} {r['t'] / 60:8.1f} {r['tokens'] / 1e6:9.1f} {np.mean([x['tok_s'] for x in tr[-10:]]):8,.0f} "
              f"{r['cosmo/bpb']:10.4f} {r.get('web/bpb', float('nan')):8.4f} {r.get('wiki/bpb', float('nan')):9.4f}")
    if "kd_muon" in final and "ce_muon" in final:
        d = final["ce_muon"] - final["kd_muon"]
        ratio = speed["ce_muon"] / speed["kd_muon"]
        print(f"\ndistillation vs plain CE at equal time: {d:+.4f} bits/byte ({'KD ahead' if d > 0 else 'CE ahead'}). "
              + ("Keep KD = True." if d > 0.01 else f"Plain CE is competitive: consider KD = False (it trains {ratio:.2f}x more tokens per hour)."))
    if "kd_muon" in final and "kd_adamw" in final:
        d = final["kd_adamw"] - final["kd_muon"]
        print(f"Muon vs AdamW at equal time: {d:+.4f} bits/byte ({'Muon ahead' if d > 0 else 'AdamW ahead'}). "
              + ("Keep USE_MUON = True." if d > -0.005 else "Set USE_MUON = False."))

## 4. Stage 1: pretraining (distillation on the mix)

Resumes automatically from `pretrain_latest.pt`. `pretrain_best.pt` is kept whenever the mean held-out bits-per-byte (Cosmopedia + FineWeb-Edu) improves. At `MAX_RUN_HOURS` it saves and stops cleanly (**PAUSED**): start a new session, attach this version's output as an input, and Run All again with the same settings.

In [ ]:
if ("pretrain" in STAGES) and not PAUSED:
    launch("pretrain")
    st = stage_state("pretrain")
    if st.get("status") == "paused":
        PAUSED = True
        print("=" * 100)
        print(f"PRETRAINING PAUSED at step {st['step']}/{st['total_steps']} ({st['tokens'] / 1e6:,.0f}M tokens, {st['elapsed'] / 3600:.1f} h of training so far).")
        print("The checkpoint is saved; the cells below are skipped so this session finishes normally and keeps its output.")
        print("To continue: new session, attach this version's output as an input, Run All (keep TOKENS_PER_STEP / BATCH_SIZE / the model unchanged).")
        print("=" * 100)
    else:
        print("pretraining DONE:", st)

## 5. Stage 2: instruction tuning

Starts from `pretrain_best.pt`. Every example is `### Question:\n...\n\n### Answer:\n<response><eos>`; the loss is on the answer tokens only, with `Qwen2.5-0.5B-Instruct` as the teacher (`SFT_ALPHA = 0` for plain SFT). Examples whose answers contain more than `SFT_MAX_UNK_FRAC` unknown tokens are dropped. The same optimizer family as pretraining is used.

In [ ]:
if ("sft" in STAGES) and not PAUSED:
    launch("sft")
    st = stage_state("sft")
    if st.get("status") == "paused":
        PAUSED = True
        print(f"INSTRUCTION TUNING PAUSED at step {st['step']}/{st['total_steps']}: attach this output as an input and run again.")
    else:
        print("instruction tuning DONE:", st)

## 6. Export, fp16 check, report

One small self-contained file per model: fp16 weights (the tied embedding is stored once) with the config and vocabulary in its metadata. The check loads the exported file as a **pure fp16** model and compares its held-out cross-entropy with the fp32 checkpoint's.

In [ ]:
if ("export" in STAGES) and not PAUSED:
    from safetensors import safe_open
    vocab_json = json.load(open(VOCAB_PATH))
    vocab = K.VocabMap(vocab_json)
    vocab.set_bytes(K.teacher_token_bytes(tokenizer, TEACHER_VOCAB))
    amp = nullcontext
    PRE_BEST, SFT_BEST = (os.path.join(CKPT_DIR, f) for f in ("pretrain_best.pt", "sft_best.pt"))

    out_files = {}
    for ckpt, name in ((PRE_BEST, "student_pretrained"), (SFT_BEST, "student_chat")):
        if os.path.exists(ckpt):
            out_files[name] = os.path.join(PERSIST_DIR, name + ".safetensors")
            K.export_final(ckpt, out_files[name])

    # pure-fp16 check on the pretrained model
    val = K.TokenFile(D.paths(DATA_DIR)["cosmo_val"], SEQ_LEN).fixed(min(VAL_WINDOWS, 64))
    ref, _ = K.load_checkpoint(PRE_BEST, DEVICE, vocab_json)
    with safe_open(out_files["student_pretrained"], framework="pt") as f:
        meta, tensors = f.metadata(), {k: f.get_tensor(k) for k in f.keys()}
    half = M.StudentLM(M.Config.from_dict(json.loads(meta["config"])))
    half.load_state_dict(tensors, strict=False)               # lm_head is tied to the embedding
    ce32 = K.evaluate(ref, vocab, val, DEVICE, amp, None, EVAL_BS)["ce"]
    ce16 = K.evaluate(half.half().to(DEVICE), vocab, val, DEVICE, amp, None, EVAL_BS)["ce"]
    print(f"held-out CE  fp32 checkpoint {ce32:.4f} | exported pure-fp16 model {ce16:.4f} | difference {ce16 - ce32:+.4f}")
    print("fp16 export OK" if abs(ce16 - ce32) < 0.02 else "WARNING: the fp16 export differs noticeably from fp32")

In [ ]:
if ("export" in STAGES) and not PAUSED:
    # the 10M model of the previous run, scored on the SAME held-out sets and benchmarks (bits/byte makes the vocabularies comparable)
    def find_v4():
        cands = glob.glob(os.path.join(KAGGLE_INPUT, "**", "student_pretrained.safetensors"), recursive=True)
        cands += glob.glob("kaggle_distilv4_output/**/student_pretrained.safetensors", recursive=True)
        for p in cands:
            with safe_open(p, framework="pt") as f:
                if (f.metadata() or {}).get("format") == "distil-student-v1":
                    return p


    BASELINE = {}
    p10 = find_v4()
    if p10 is None:
        print("10M baseline weights not found (attach the previous run's student_pretrained.safetensors as an input to include them in the report)")
    else:
        with safe_open(p10, framework="pt") as f:
            meta10, t10 = f.metadata(), {k: f.get_tensor(k) for k in f.keys()}
        cfg10 = M.Config.from_v4(json.loads(meta10["config"]))
        m10 = M.StudentLM(cfg10)
        m10.load_state_dict(M.from_v4_state_dict(t10, cfg10), strict=False)
        m10 = m10.to(DEVICE).eval()
        v10 = K.VocabMap(json.loads(meta10["vocab"]))
        v10.set_bytes(K.teacher_token_bytes(tokenizer, v10.raw["teacher_vocab_size"]))
        P = D.paths(DATA_DIR)
        for n in ("cosmo", "web", "wiki"):
            if os.path.exists(P[f"{n}_val"]):
                BASELINE[f"{n}/bpb"] = K.evaluate(m10, v10, K.TokenFile(P[f"{n}_val"], SEQ_LEN).fixed(VAL_WINDOWS), DEVICE, amp, None, EVAL_BS)["bpb"]
        raw = K.read_json(P["bench"])
        if raw and BENCH_EVERY > 0:
            BASELINE.update(K.run_bench(m10, v10, K.prep_bench(raw, tokenizer), DEVICE, amp, SEQ_LEN - 1))
        print(f"10M baseline ({p10}):", {k: round(v, 4) for k, v in BASELINE.items()})
        del m10

In [ ]:
if ("export" in STAGES) and not PAUSED:
    # the report: everything worth putting in a write-up, in one JSON file
    def last(h, kind):
        return (h.get(kind) or [{}])[-1]

    hp, hs = K.read_json(os.path.join(CKPT_DIR, "history_pretrain.json"), {}), K.read_json(os.path.join(CKPT_DIR, "history_sft.json"), {})
    info = K.read_json(os.path.join(CKPT_DIR, "info_pretrain.json"), {})
    sp, ss = stage_state("pretrain"), stage_state("sft")
    ev, be = last(hp, "eval"), last(hp, "bench")
    hours = lambda s: s.get("elapsed", 0) * s.get("world", 1) / 3600
    report = dict(
        model=dict(params=info.get("params"), config=info.get("config"), gpu=info.get("gpu"), n_gpu=info.get("world"), precision=info.get("precision"),
                   kd=info.get("kd"), muon=info.get("muon")),
        pretrain=dict(tokens=sp.get("tokens"), steps=sp.get("step"), tokens_per_param=(sp.get("tokens") or 0) / max(1, info.get("params") or 1),
                      wall_hours=sp.get("elapsed", 0) / 3600, gpu_hours=hours(sp), final_eval=ev, bench=be,
                      tok_s=float(np.mean([r["tok_s"] for r in hp.get("train", [])[-20:]] or [0]))),
        sft=dict(steps=ss.get("step"), gpu_hours=hours(ss), final_eval=last(hs, "eval"), bench=last(hs, "bench")),
        baseline_10m=BASELINE,
        gpu_hours_total=hours(sp) + hours(ss),
    )
    json.dump(report, open(os.path.join(PERSIST_DIR, "report.json"), "w"), indent=1, default=float)
    rows = [("held-out bits/byte  cosmo", "cosmo/bpb"), ("                    web", "web/bpb"), ("                    wiki (OOD)", "wiki/bpb"),
            ("ARC-Easy acc_norm %", "arc_easy_norm"), ("HellaSwag acc_norm %", "hellaswag_norm"), ("SciQ acc %", "sciq"), ("LAMBADA acc %", "lambada_acc")]
    print(f"{'':28s} {'10M (previous)':>15s} {'51M (this run)':>15s}")
    for label, key in rows:
        a = BASELINE.get(key)
        b = ev.get(key, be.get(key))
        print(f"{label:28s} {('%.4f' % a if a is not None else '-'):>15s} {('%.4f' % b if b is not None else '-'):>15s}")
    print(f"\n{info.get('params', 0) / 1e6:.1f}M parameters | {sp.get('tokens', 0) / 1e9:.2f}B tokens | {hours(sp):.1f} GPU-hours pretraining + {hours(ss):.1f} instruction tuning")
    print("saved", os.path.join(PERSIST_DIR, "report.json"))

## 7. Chat with the student

In [ ]:
if ("export" in STAGES) and not PAUSED:
    chat_ckpt = SFT_BEST if os.path.exists(SFT_BEST) else PRE_BEST
    chat_model, _ = K.load_checkpoint(chat_ckpt, DEVICE, vocab_json)
    chat_model.eval()


    def ask(query, max_new_tokens=200, temperature=0.6, top_k=40):
        return K.sample_text(chat_model, vocab, tokenizer, K.PROMPT_TEMPLATE.format(query=query), DEVICE, amp, max_new_tokens, temperature, top_k, 1.15).strip()


    print("chatting with", os.path.basename(chat_ckpt))
    for q in ["What is the capital of France?", "Why do we need to sleep?", "Explain how rain forms."]:
        print("Q:", q, "\nA:", ask(q), "\n")